# DataInsight — Time series: Dự báo lưu lượng giao thông theo giờ trên cao tốc I-94

| | |
|---|---|
| **Học phần** | Nền tảng lập trình cho phân tích và trực quan dữ liệu (CO5177) — HK 261, 2026–2027 |
| **Giảng viên** | Lê Thành Sách — Khoa KH&KT Máy tính, ĐH Bách Khoa – ĐHQG-HCM |
| **Nhóm** | DataInsight |
| **Phụ trách bài Time series** | Trương Thanh Minh (2580893) |
| **Dataset** | [Metro Interstate Traffic Volume](https://www.kaggle.com/datasets/pooriamst/metro-interstate-traffic-volume) — Kaggle, bản gốc UCI, giấy phép CC BY 4.0 |

**Bài toán.** Dự báo `traffic_volume` (số xe/giờ đi hướng tây trên I-94, đoạn Minneapolis – St Paul) từ lịch sử lưu lượng, lịch (giờ, thứ, ngày lễ) và thời tiết.

Notebook này gồm **phần 1 – mô tả dữ liệu, EDA và chuẩn bị dữ liệu**. Phần mô hình hóa sẽ nối tiếp sau mục 6, dùng lại các biến `X_train`, `X_val`, `X_test` tạo ở cuối notebook.

**Mục lục**
1. [Cài đặt và tải dữ liệu](#1)
2. [Mô tả tập dữ liệu](#2)
3. [Kiểm tra chất lượng dữ liệu](#3)
4. [Làm sạch: gộp trùng, chọn giai đoạn, reindex và nội suy](#4)
5. [EDA trực quan](#5)
   - 5.1 Toàn cảnh chuỗi · 5.2 Phân phối · 5.3 Mùa vụ ngày/tuần/năm · 5.4 Ngày lễ · 5.5 Thời tiết
   - 5.6 Phân rã mùa vụ (MSTL) · 5.7 Tự tương quan (ACF/PACF) · 5.8 Tính dừng · 5.9 Ngoại lai
6. [Chuẩn bị dữ liệu cho mô hình: đặc trưng, chia theo thời gian, encoding, scaling](#6)
7. [Tổng kết EDA](#7)

> Notebook chạy được bằng **Runtime → Run all** trên Google Colab: dữ liệu tải thẳng từ UCI Machine Learning Repository, không cần Kaggle API key hay mount Drive.
> Biểu đồ dùng Plotly (tương tác khi mở bằng Colab/Jupyter).

<a id="1"></a>
## 1. Cài đặt và tải dữ liệu

### 1.1 Thư viện

Colab đã có sẵn `pandas`, `numpy`, `scikit-learn`; ô dưới chỉ bảo đảm `plotly` và `statsmodels` đủ mới (cần `MSTL` của statsmodels ≥ 0.14).

In [1]:
%pip install -q "plotly>=5.15" "statsmodels>=0.14"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from statsmodels.tsa.seasonal import MSTL
from statsmodels.tsa.stattools import acf, adfuller, kpss, pacf

In [3]:
warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

pio.renderers.default = "colab" if "google.colab" in sys.modules else "plotly_mimetype"

### 1.2 Bảng màu và theme dùng chung

Màu gán theo **vai trò** và giữ cố định trong toàn notebook: xanh dương = chuỗi chính / ngày thường, cam = cuối tuần / so sánh, xanh ngọc = ngày lễ, đỏ = giảm / ngoại lai, xám = tham chiếu.

In [4]:
BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
RED, MUTED, GRID, INK_2 = "#e34948", "#898781", "#e1e0d9", "#52514e"
SEQ_BLUE = ["#f4f8fd", "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
DIVERGING = [[0, BLUE], [0.5, "#f0efec"], [1, RED]]

pio.templates["datainsight"] = go.layout.Template(layout=dict(
    font=dict(family="Inter, Segoe UI, Arial, sans-serif", size=13, color="#0b0b0b"),
    colorway=[BLUE, ORANGE, AQUA, YELLOW],
    paper_bgcolor="#ffffff", plot_bgcolor="#ffffff",
    title=dict(x=0.01, font=dict(size=16)),
    xaxis=dict(gridcolor=GRID, linecolor="#c9c8c3", zeroline=False, title_font_color=INK_2),
    yaxis=dict(gridcolor=GRID, linecolor="#c9c8c3", zeroline=False, title_font_color=INK_2),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0, title_text=""),
    margin=dict(l=60, r=20, t=80, b=50),
))
pio.templates.default = "plotly_white+datainsight"

DAY_LABELS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
MONTH_LABELS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

### 1.3 Hàm vẽ dùng lại

Các biểu đồ phân phối được **tính sẵn bằng numpy** (histogram, phân vị) thay vì đưa hàng chục nghìn điểm vào Plotly — notebook nhẹ hơn nhiều và mở nhanh hơn.

In [5]:
def hist_bar(values, bins, color=BLUE, name=None):
    """Histogram tính sẵn → go.Bar."""
    counts, edges = np.histogram(values[~np.isnan(values)], bins=bins)
    centers = (edges[:-1] + edges[1:]) / 2
    return go.Bar(x=centers, y=counts, width=np.diff(edges) * 0.92, marker_color=color, name=name,
                  hovertemplate="%{x:,.1f}: %{y:,} hours<extra></extra>")


def box_from_groups(series, groups, color=BLUE, name=None):
    """Box plot từ phân vị tính sẵn của từng nhóm (không nhúng điểm dữ liệu thô)."""
    q = series.groupby(groups).quantile([0.05, 0.25, 0.5, 0.75, 0.95]).unstack()
    return go.Box(x=q.index, q1=q[0.25], median=q[0.5], q3=q[0.75], lowerfence=q[0.05], upperfence=q[0.95],
                  marker_color=color, line_width=1.5, name=name)


def band(fig, x, lo, hi, color, name, row=None, col=None):
    """Dải tô màu giữa hai đường lo–hi."""
    kw = dict(row=row, col=col) if row else {}
    fig.add_trace(go.Scatter(x=np.r_[x, x[::-1]], y=np.r_[hi, lo[::-1]], fill="toself", fillcolor=color,
                             line=dict(width=0), hoverinfo="skip", name=name), **kw)

### 1.4 Tải dữ liệu

In [6]:
DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00492/Metro_Interstate_Traffic_Volume.csv.gz"

# keep_default_na=False: giữ chuỗi "None" của cột holiday, không để pandas đổi thành NaN.
raw = pd.read_csv(DATA_URL, compression="gzip", keep_default_na=False, na_values=[])
raw["date_time"] = pd.to_datetime(raw["date_time"])
print("Đã tải dữ liệu từ:", DATA_URL)
print(f"Kích thước: {raw.shape[0]:,} dòng × {raw.shape[1]} cột")

Đã tải dữ liệu từ: https://archive.ics.uci.edu/ml/machine-learning-databases/00492/Metro_Interstate_Traffic_Volume.csv.gz
Kích thước: 48,204 dòng × 9 cột


In [7]:
raw.head()

,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,None,288.28,0.00,0.00,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,None,289.36,0.00,0.00,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,None,289.58,0.00,0.00,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,None,290.13,0.00,0.00,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,None,291.14,0.00,0.00,75,Clouds,broken clouds,2012-10-02 13:00:00,4918


<a id="2"></a>
## 2. Mô tả tập dữ liệu

Dữ liệu ghi lưu lượng xe theo giờ tại trạm đếm ATR 301 của Sở Giao thông Minnesota (MnDOT), nằm khoảng giữa Minneapolis và St Paul, kèm thời tiết theo giờ (OpenWeatherMap) và ngày lễ. Khoảng thời gian: 10/2012 – 09/2018.

| Cột | Kiểu | Ý nghĩa |
|---|---|---|
| `date_time` | datetime | Mốc giờ (giờ địa phương CST) |
| `traffic_volume` | số nguyên | **Biến mục tiêu** — số xe đi hướng tây trong giờ đó |
| `holiday` | phân loại | Tên ngày lễ quốc gia/vùng (`None` nếu không phải ngày lễ) |
| `temp` | số thực | Nhiệt độ trung bình (Kelvin) |
| `rain_1h`, `snow_1h` | số thực | Lượng mưa / tuyết trong giờ (mm) |
| `clouds_all` | số nguyên | Độ che phủ mây (%) |
| `weather_main` | phân loại | Nhóm thời tiết (Clear, Clouds, Rain, Snow, ...) |
| `weather_description` | phân loại | Mô tả thời tiết chi tiết hơn |

### 2.1 Kiểu dữ liệu và thống kê mô tả

In [8]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48204 entries, 0 to 48203
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   holiday              48204 non-null  object        
 1   temp                 48204 non-null  float64       
 2   rain_1h              48204 non-null  float64       
 3   snow_1h              48204 non-null  float64       
 4   clouds_all           48204 non-null  int64         
 5   weather_main         48204 non-null  object        
 6   weather_description  48204 non-null  object        
 7   date_time            48204 non-null  datetime64[ns]
 8   traffic_volume       48204 non-null  int64         
dtypes: datetime64[ns](1), float64(3), int64(2), object(3)
memory usage: 3.3+ MB


In [9]:
raw.describe().T

,count,mean,min,25%,50%,75%,max,std
temp,"48,204.00",281.21,0.00,272.16,282.45,291.81,310.07,13.34
rain_1h,"48,204.00",0.33,0.00,0.00,0.00,0.00,"9,831.30",44.79
snow_1h,"48,204.00",0.00,0.00,0.00,0.00,0.00,0.51,0.01
clouds_all,"48,204.00",49.36,0.00,1.00,64.00,90.00,100.00,39.02
date_time,48204,2016-01-05 10:46:16.773711616,2012-10-02 09:00:00,2014-02-06 11:45:00,2016-06-11 03:30:00,2017-08-11 06:00:00,2018-09-30 23:00:00,NaN
traffic_volume,"48,204.00","3,259.82",0.00,"1,193.00","3,380.00","4,933.00","7,280.00","1,986.86"


Đã thấy hai dấu hiệu bất thường ngay trong bảng thống kê: `temp` có **min = 0 K** (−273 °C) và `rain_1h` có **max ≈ 9.831 mm** trong một giờ. Cả hai được kiểm tra ở mục 3.3.

### 2.2 Các cột phân loại

In [10]:
cat_summary = pd.DataFrame({
    "Số giá trị khác nhau": raw[["holiday", "weather_main", "weather_description"]].nunique(),
    "Giá trị phổ biến nhất": raw[["holiday", "weather_main", "weather_description"]].mode().iloc[0],
})
cat_summary

,Số giá trị khác nhau,Giá trị phổ biến nhất
holiday,12,None
weather_main,11,Clouds
weather_description,38,sky is clear


In [11]:
holiday_counts = raw.loc[raw["holiday"] != "None", "holiday"].value_counts().rename("Số dòng").to_frame()
holiday_counts

,Số dòng
holiday,
Labor Day,7
Christmas Day,6
Thanksgiving Day,6
Martin Luther King Jr Day,6
New Years Day,6
Veterans Day,5
Columbus Day,5
Memorial Day,5
Washingtons Birthday,5


In [12]:
wd = raw.groupby(["weather_main", "weather_description"]).size().rename("n").reset_index()
wm = wd.groupby("weather_main")["n"].sum().sort_values(ascending=False)
palette = [BLUE, ORANGE, AQUA, YELLOW, RED, "#7b5fd6", "#3fa7c9", "#b4782f", MUTED, "#d65f9e", "#5e8f3a"]
color_of = dict(zip(wm.index, palette))

fig = go.Figure(go.Sunburst(
    ids=list(wm.index) + [f"{m}/{d}" for m, d in zip(wd["weather_main"], wd["weather_description"])],
    labels=list(wm.index) + list(wd["weather_description"]),
    parents=[""] * len(wm) + list(wd["weather_main"]),
    values=list(wm.values) + list(wd["n"]),
    branchvalues="total",
    marker=dict(colors=[color_of[m] for m in wm.index] + [color_of[m] for m in wd["weather_main"]],
                line=dict(color="#ffffff", width=1.5)),
    insidetextorientation="radial",
    hovertemplate="%{label}: %{value:,} rows (%{percentRoot:.1%})<extra></extra>",
))
fig.update_layout(title="Figure 1. Weather types (inner ring) and detailed descriptions (outer ring), raw data",
                  height=620, margin=dict(t=70, l=10, r=10, b=10))
fig.show()

**Nhận xét.** 11 nhóm `weather_main` được chia nhỏ thành 38 mô tả. `Clouds` và `Clear` chiếm phần lớn số dòng; các nhóm như `Squall`, `Smoke` chỉ có vài chục dòng → khi làm đặc trưng sẽ gộp nhóm hiếm vào `Other`. Ngày lễ chỉ có **61 dòng** trên 48 nghìn — sẽ thấy ở mục 3.4 lý do là nhãn chỉ gắn vào một giờ duy nhất của ngày lễ.

<a id="3"></a>
## 3. Kiểm tra chất lượng dữ liệu

Dữ liệu chuỗi thời gian theo giờ cần đúng **một dòng cho mỗi giờ, không thiếu giờ nào**. Mục này kiểm tra bốn vấn đề: mốc giờ trùng, mốc giờ bị thiếu, giá trị cảm biến vô lý và cách gán nhãn ngày lễ.

### 3.1 Mốc giờ bị trùng

In [13]:
dup_mask = raw.duplicated("date_time", keep=False)
n_dup = raw.duplicated("date_time").sum()
print(f"Số dòng thừa do trùng mốc giờ: {n_dup:,} / {len(raw):,} ({n_dup / len(raw):.1%})")
print(f"Số mốc giờ duy nhất: {raw['date_time'].nunique():,}")

Số dòng thừa do trùng mốc giờ: 7,629 / 48,204 (15.8%)
Số mốc giờ duy nhất: 40,575


Trong mỗi nhóm dòng trùng mốc giờ, cột nào thực sự khác nhau?

In [14]:
n_distinct = raw[dup_mask].groupby("date_time").nunique()
differs = (n_distinct > 1).mean().mul(100).sort_values()

fig = go.Figure(go.Bar(x=differs.values, y=differs.index, orientation="h",
                       marker_color=np.where(differs.values > 50, ORANGE, MUTED),
                       text=[f"{v:.1f}%" for v in differs.values], textposition="outside", textfont_color=INK_2,
                       hovertemplate="%{y}: %{x:.1f}% of duplicate groups differ<extra></extra>"))
fig.update_layout(title="Figure 2. Within duplicated timestamps, only the weather columns differ",
                  xaxis_title="% of duplicate groups where the column has >1 distinct value",
                  xaxis_range=[0, 112], height=380, showlegend=False)
fig.show()

In [15]:
example_time = raw.loc[dup_mask, "date_time"].value_counts().idxmax()
raw.loc[raw["date_time"] == example_time]

,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
6162,None,287.15,0.00,0.00,90,Thunderstorm,thunderstorm with light rain,2013-05-19 10:00:00,3591
6163,None,287.15,0.00,0.00,90,Mist,mist,2013-05-19 10:00:00,3591
6164,None,287.15,0.00,0.00,90,Thunderstorm,proximity thunderstorm,2013-05-19 10:00:00,3591
6165,None,287.15,0.00,0.00,90,Rain,moderate rain,2013-05-19 10:00:00,3591
6166,None,287.15,0.00,0.00,90,Rain,light rain,2013-05-19 10:00:00,3591
6167,None,287.15,0.00,0.00,90,Thunderstorm,thunderstorm with heavy rain,2013-05-19 10:00:00,3591


**Nhận xét.** Các dòng trùng chỉ khác nhau ở `weather_main` / `weather_description`: khi trong một giờ có nhiều hiện tượng thời tiết (vd. vừa mưa nhỏ vừa sương mù), nguồn dữ liệu ghi mỗi hiện tượng thành một dòng và **lặp lại nguyên `traffic_volume`**. Nếu giữ nguyên, các giờ thời tiết phức tạp bị đếm nhiều lần và chuỗi không còn đều theo giờ. → Ở mục 4 ta **gộp về một dòng mỗi giờ**, giữ hiện tượng đầu tiên và thêm cột `n_weather` (số hiện tượng thời tiết trong giờ) để không mất thông tin.

### 3.2 Mốc giờ bị thiếu

Đưa chuỗi về tần suất đúng 1 giờ (`asfreq("h")`): giờ nào không có dòng sẽ hiện ra dưới dạng `NaN`.

In [16]:
hourly_raw = raw.drop_duplicates("date_time").set_index("date_time").sort_index().asfreq("h")
is_missing = hourly_raw["traffic_volume"].isna()
print(f"Từ {hourly_raw.index.min()} đến {hourly_raw.index.max()}")
print(f"Số giờ theo lịch: {len(hourly_raw):,} | có dữ liệu: {(~is_missing).sum():,} | "
      f"thiếu: {is_missing.sum():,} ({is_missing.mean():.1%})")

Từ 2012-10-02 09:00:00 đến 2018-09-30 23:00:00
Số giờ theo lịch: 52,551 | có dữ liệu: 40,575 | thiếu: 11,976 (22.8%)


Mỗi đoạn liên tiếp các giờ thiếu gọi là một **gap**. Hàm dưới gom các giờ thiếu thành từng gap (bắt đầu, kết thúc, độ dài).

In [17]:
def gap_table(missing):
    run_id = missing.ne(missing.shift()).cumsum()[missing]
    idx = pd.Series(missing.index[missing], index=run_id.values)
    return (idx.groupby(level=0).agg(start="min", end="max", hours="size")
              .sort_values("hours", ascending=False).reset_index(drop=True))


gaps_all = gap_table(is_missing)
print(f"Số đoạn bị thiếu: {len(gaps_all):,}")
gaps_all.head(6)

Số đoạn bị thiếu: 2,588


,start,end,hours
0,2014-08-08 02:00:00,2015-06-11 19:00:00,7386
1,2013-10-27 02:00:00,2013-11-06 03:00:00,242
2,2015-06-14 21:00:00,2015-06-19 17:00:00,117
3,2014-04-29 09:00:00,2014-05-04 04:00:00,116
4,2015-10-23 12:00:00,2015-10-27 07:00:00,92
5,2014-07-27 17:00:00,2014-07-30 08:00:00,64


In [18]:
daily_raw = hourly_raw["traffic_volume"].resample("D").mean()
big = gaps_all.iloc[0]

fig = go.Figure(go.Scatter(x=daily_raw.index, y=daily_raw.values, mode="lines", line=dict(color=BLUE, width=1.3),
                           name="Daily mean", hovertemplate="%{x|%d/%m/%Y}: %{y:,.0f} vehicles/hour<extra></extra>"))
fig.add_vrect(x0=big.start, x1=big.end, fillcolor=ORANGE, opacity=0.12, line_width=0,
              annotation_text=f"{big.hours:,} consecutive hours missing (~{big.hours / 24:.0f} days)",
              annotation_position="top left", annotation_font_color=INK_2)
fig.add_vrect(x0=pd.Timestamp("2016-01-01"), x1=daily_raw.index.max(), fillcolor=BLUE, opacity=0.06, line_width=0,
              annotation_text="Period selected for modeling", annotation_position="top right",
              annotation_font_color=BLUE)
fig.update_layout(title="Figure 3. Daily mean traffic volume, 2012–2018: the series has many breaks before 2016",
                  yaxis_title="Vehicles/hour (daily mean)", height=420, showlegend=False)
fig.show()

In [19]:
coverage = (~is_missing).resample("MS").mean() * 100
cov_grid = coverage.to_frame("pct").assign(year=coverage.index.year, month=coverage.index.month)
cov_grid = cov_grid.pivot(index="year", columns="month", values="pct").reindex(columns=range(1, 13))

fig = go.Figure(go.Heatmap(z=cov_grid.values, x=MONTH_LABELS, y=cov_grid.index.astype(str),
                           colorscale=[[0, RED], [0.5, "#f6d7c3"], [0.9, "#cde2fb"], [1, BLUE]], zmin=0, zmax=100,
                           text=[["" if np.isnan(v) else f"{v:.0f}" for v in row] for row in cov_grid.values],
                           texttemplate="%{text}", textfont=dict(size=11), xgap=3, ygap=3,
                           colorbar=dict(title="% hours<br>observed"),
                           hovertemplate="%{x} %{y}: %{z:.1f}% of hours observed<extra></extra>"))
fig.update_layout(title="Figure 4. Data coverage by month: 2014–2015 has a ~10-month hole, 2016–2018 is almost complete",
                  height=380, yaxis_autorange="reversed")
fig.show()

In [20]:
coverage_by_period = pd.DataFrame({
    "Giai đoạn": ["10/2012 – 12/2015", "01/2016 – 09/2018"],
    "% giờ có dữ liệu": [100 * (~is_missing[:"2015-12-31"]).mean(), 100 * (~is_missing["2016-01-01":]).mean()],
    "Gap dài nhất (giờ)": [gap_table(is_missing[:"2015-12-31"])["hours"].max(),
                           gap_table(is_missing["2016-01-01":])["hours"].max()],
})
coverage_by_period

,Giai đoạn,% giờ có dữ liệu,Gap dài nhất (giờ)
0,10/2012 – 12/2015,61.47,7386
1,01/2016 – 09/2018,95.80,9


In [21]:
bins = [0, 1, 3, 6, 24, 168, np.inf]
labels = ["1 hour", "2–3 hours", "4–6 hours", "7–24 hours", "1–7 days", "> 7 days"]
gap_buckets = pd.cut(gaps_all["hours"], bins=bins, labels=labels).value_counts().reindex(labels)
hours_lost = gaps_all.groupby(pd.cut(gaps_all["hours"], bins=bins, labels=labels), observed=False)["hours"].sum()

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
                    subplot_titles=("Number of gaps", "Hours lost (share of all missing hours)"))
fig.add_trace(go.Bar(x=labels, y=gap_buckets.values, marker_color=BLUE, text=gap_buckets.values,
                     textposition="outside", textfont_color=INK_2,
                     hovertemplate="%{x}: %{y:,} gaps<extra></extra>"), row=1, col=1)
fig.add_trace(go.Bar(x=labels, y=hours_lost.values, marker_color=ORANGE,
                     text=[f"{v / hours_lost.sum():.0%}" for v in hours_lost.values],
                     textposition="outside", textfont_color=INK_2,
                     hovertemplate="%{x}: %{y:,} hours<extra></extra>"), row=1, col=2)
fig.update_yaxes(range=[0, gap_buckets.max() * 1.18], row=1, col=1)
fig.update_yaxes(range=[0, hours_lost.max() * 1.18], row=1, col=2)
fig.update_layout(title="Figure 5. Most gaps last 1 hour, but most missing hours come from a few gaps longer than 7 days",
                  height=420, showlegend=False)
fig.show()

**Nhận xét.** Đa số gap chỉ dài 1 giờ (trạm đếm mất tín hiệu ngắn) — loại này nội suy an toàn. Nhưng tính theo **số giờ bị mất**, phần lớn đến từ vài gap dài hơn 7 ngày trước 2016, đặc biệt đoạn gần **10 tháng** (08/2014 – 06/2015): không phương pháp nội suy nào phục hồi được, và lag theo tuần (`lag_168`) sẽ bị sai trên các đoạn đó.

→ **Quyết định:** dùng giai đoạn **01/01/2016 – 30/09/2018** (≈ 2 năm 9 tháng, đủ 2 chu kỳ năm cho mùa vụ). Trong giai đoạn này đoạn thiếu dài nhất chỉ 9 giờ (bảng trên), nên reindex + nội suy theo thời gian là hợp lý.

### 3.3 Giá trị cảm biến vô lý

In [22]:
print("Nhiệt độ thấp nhất (K):", raw["temp"].min(), "→ 0 K = −273 °C, chắc chắn là lỗi cảm biến")
raw.loc[raw["temp"] <= 0, ["date_time", "temp", "weather_main"]]

Nhiệt độ thấp nhất (K): 0.0 → 0 K = −273 °C, chắc chắn là lỗi cảm biến


,date_time,temp,weather_main
11898,2014-01-31 03:00:00,0.00,Clear
11899,2014-01-31 04:00:00,0.00,Clear
11900,2014-01-31 05:00:00,0.00,Clear
11901,2014-01-31 06:00:00,0.00,Clear
11946,2014-02-02 03:00:00,0.00,Clear
11947,2014-02-02 04:00:00,0.00,Clear
11948,2014-02-02 05:00:00,0.00,Clear
11949,2014-02-02 06:00:00,0.00,Clear
11950,2014-02-02 07:00:00,0.00,Clear
11951,2014-02-02 08:00:00,0.00,Clear


In [23]:
print("Lượng mưa lớn nhất trong 1 giờ (mm):", raw["rain_1h"].max(), "→ kỷ lục thế giới chỉ khoảng 300 mm/giờ")
raw.nlargest(3, "rain_1h")[["date_time", "rain_1h", "weather_description"]]

Lượng mưa lớn nhất trong 1 giờ (mm): 9831.3 → kỷ lục thế giới chỉ khoảng 300 mm/giờ


,date_time,rain_1h,weather_description
24872,2016-07-11 17:00:00,"9,831.30",very heavy rain
8247,2013-08-07 02:00:00,55.63,very heavy rain
7179,2013-06-24 11:00:00,44.45,very heavy rain


In [24]:
temp_c_raw = raw["temp"].to_numpy() - 273.15
rain_pos = raw.loc[raw["rain_1h"] > 0, "rain_1h"].to_numpy()

fig = make_subplots(rows=1, cols=2, subplot_titles=("Temperature (°C) — 10 hours at −273 °C",
                                                    "Hourly rainfall (mm, log scale)"))
fig.add_trace(hist_bar(temp_c_raw, bins=80, color=BLUE), row=1, col=1)
fig.add_trace(hist_bar(np.log10(rain_pos), bins=60, color=AQUA), row=1, col=2)
fig.add_annotation(x=-273, y=1, text="sensor error", showarrow=True, arrowcolor=RED, ay=-40,
                   font_color=RED, row=1, col=1)
fig.add_annotation(x=np.log10(rain_pos.max()), y=0, text="9,831 mm", showarrow=True, arrowcolor=RED, ay=-40,
                   font_color=RED, row=1, col=2)
fig.update_xaxes(title_text="°C", row=1, col=1)
fig.update_xaxes(title_text="log10(mm)", row=1, col=2)
fig.update_yaxes(title_text="Hours (log scale)", type="log", dtick=1)
fig.update_layout(title="Figure 6. Two weather columns contain outliers caused by sensor errors", showlegend=False, height=400)
fig.show()

**Nhận xét.** 10 giờ có `temp = 0 K` và 1 giờ có `rain_1h ≈ 9.831 mm` là **lỗi đo**, không phải hiện tượng thật → đổi thành `NaN` rồi nội suy như giờ bị thiếu. Ngoài ra đổi nhiệt độ sang °C cho dễ đọc. Các ngoại lai của chính `traffic_volume` (ví dụ đường bị đóng) là sự kiện có thật nên được xử lý riêng ở mục 5.9.

### 3.4 Nhãn ngày lễ chỉ gắn ở 00:00

In [25]:
holiday_rows = raw[raw["holiday"] != "None"]
print("Giờ trong ngày của các dòng có nhãn holiday:")
print(holiday_rows["date_time"].dt.hour.value_counts().to_string())

Giờ trong ngày của các dòng có nhãn holiday:
date_time
0    61


In [26]:
holiday_rows.head(8)[["date_time", "holiday", "traffic_volume"]]

,date_time,holiday,traffic_volume
126,2012-10-08,Columbus Day,455
1123,2012-11-12,Veterans Day,1000
1370,2012-11-22,Thanksgiving Day,919
2360,2012-12-25,Christmas Day,803
2559,2013-01-01,New Years Day,1439
3697,2013-02-18,Washingtons Birthday,556
6430,2013-05-27,Memorial Day,863
7414,2013-07-04,Independence Day,1060


**Nhận xét.** Nhãn ngày lễ chỉ xuất hiện ở dòng **00:00**, còn 23 giờ còn lại của cùng ngày ghi `None`. Nếu dùng nguyên, mô hình sẽ hiểu sai rằng "ngày lễ" chỉ kéo dài 1 giờ → ta **lan nhãn ra cả ngày** dựa vào ngày của mốc 00:00.

<a id="4"></a>
## 4. Làm sạch: gộp trùng, chọn giai đoạn, reindex và nội suy

Các bước, theo thứ tự:
1. Gộp các dòng trùng mốc giờ, thêm `n_weather`.
2. Đổi lỗi cảm biến (`temp ≤ 0 K`, `rain_1h > 100 mm`) thành `NaN`; đổi nhiệt độ sang °C.
3. Cắt giai đoạn 01/2016 – 09/2018 và **reindex theo tần suất 1 giờ** để lộ ra các giờ thiếu.
4. Đánh dấu `is_imputed = 1` cho giờ được điền, rồi **nội suy tuyến tính theo thời gian** các cột số. Lý do: đoạn thiếu dài nhất chỉ 9 giờ, lưu lượng và thời tiết thay đổi liên tục nên nội suy tuyến tính sát thực tế hơn forward-fill (forward-fill sẽ "đóng băng" lưu lượng lúc giờ cao điểm bắt đầu tăng/giảm).
5. Cột phân loại (`weather_main`, `weather_description`) dùng **forward-fill**: thời tiết của giờ trước là ước lượng hợp lý nhất, và không thể nội suy giữa hai nhãn.
6. Lan nhãn ngày lễ ra cả ngày.

In [27]:
WINDOW_START, WINDOW_END = "2016-01-01 00:00", "2018-09-30 23:00"
NUM_COLS = ["traffic_volume", "temp_c", "rain_1h", "snow_1h", "clouds_all"]
CLEAN_COLS = ["traffic_volume", "is_imputed", "holiday", "is_holiday", "temp_c", "rain_1h", "snow_1h",
              "clouds_all", "weather_main", "weather_description", "n_weather"]

In [28]:
def clean(raw):
    df = raw.copy()
    # Bước 1: gộp trùng
    df["n_weather"] = df.groupby("date_time")["weather_main"].transform("size")
    df = df.drop_duplicates("date_time", keep="first").set_index("date_time").sort_index()

    # Bước 2: lỗi cảm biến → NaN, đổi °C
    df.loc[df["temp"] <= 0, "temp"] = np.nan
    df.loc[df["rain_1h"] > 100, "rain_1h"] = np.nan
    df["temp_c"] = df["temp"] - 273.15

    # Ghi nhớ tên ngày lễ theo ngày (lấy từ mốc 00:00) trước khi reindex
    holiday_by_day = df.loc[df["holiday"] != "None", "holiday"]
    holiday_by_day.index = holiday_by_day.index.normalize()

    # Bước 3–5: cắt giai đoạn, reindex, nội suy / forward-fill
    df = df.loc[WINDOW_START:WINDOW_END].asfreq("h")
    df["is_imputed"] = df["traffic_volume"].isna().astype(int)
    df[NUM_COLS] = df[NUM_COLS].interpolate(method="time", limit_direction="both")
    df[["weather_main", "weather_description"]] = df[["weather_main", "weather_description"]].ffill().bfill()
    df["n_weather"] = df["n_weather"].fillna(1).astype(int)

    # Bước 6: lan nhãn ngày lễ ra cả ngày
    df["holiday"] = df.index.normalize().map(holiday_by_day).fillna("None")
    df["is_holiday"] = (df["holiday"] != "None").astype(int)
    return df[CLEAN_COLS]

In [29]:
ts = clean(raw)
y = ts["traffic_volume"]
print("Còn NaN sau làm sạch:", int(ts.isna().sum().sum()))
print("Số ngày lễ trong giai đoạn:", ts.loc[ts["is_holiday"] == 1].index.normalize().nunique())
ts.head()

Còn NaN sau làm sạch: 0
Số ngày lễ trong giai đoạn: 28


,traffic_volume,is_imputed,holiday,is_holiday,temp_c,rain_1h,snow_1h,clouds_all,weather_main,weather_description,n_weather
date_time,,,,,,,,,,,
2016-01-01 00:00:00,"1,513.00",0,New Years Day,1,-7.21,0.00,0.00,90.00,Haze,haze,2
2016-01-01 01:00:00,"1,550.00",0,New Years Day,1,-7.15,0.00,0.00,90.00,Snow,light snow,1
2016-01-01 02:00:00,"1,134.50",1,New Years Day,1,-7.14,0.00,0.00,90.00,Snow,light snow,1
2016-01-01 03:00:00,719.00,0,New Years Day,1,-7.14,0.00,0.00,90.00,Snow,light snow,1
2016-01-01 04:00:00,533.00,0,New Years Day,1,-8.35,0.00,0.00,90.00,Clouds,overcast clouds,1


In [30]:
summary = pd.DataFrame({
    "Bước": ["Dữ liệu gốc", "Sau khi gộp trùng (toàn bộ 2012–2018)", "Giờ có dữ liệu trong 2016–2018",
             "Sau reindex 2016–2018 (đủ mọi giờ)", "  trong đó: giờ được nội suy"],
    "Số dòng": [len(raw), raw["date_time"].nunique(), int((ts["is_imputed"] == 0).sum()),
                len(ts), int(ts["is_imputed"].sum())],
})
summary["% so với sau reindex"] = (summary["Số dòng"] / len(ts) * 100).round(1).astype(str) + "%"
summary.loc[[0, 1], "% so với sau reindex"] = "—"
summary

,Bước,Số dòng,% so với sau reindex
0,Dữ liệu gốc,48204,—
1,Sau khi gộp trùng (toàn bộ 2012–2018),40575,—
2,Giờ có dữ liệu trong 2016–2018,23084,95.8%
3,Sau reindex 2016–2018 (đủ mọi giờ),24096,100.0%
4,trong đó: giờ được nội suy,1012,4.2%


In [31]:
fig = go.Figure(go.Funnel(
    y=["Raw rows", "Unique hours (2012–2018)", "Observed hours in 2016–2018",
       f"Final hourly series 2016–2018<br>(+{int(ts['is_imputed'].sum()):,} interpolated hours)"],
    x=[len(raw), raw["date_time"].nunique(), int((ts["is_imputed"] == 0).sum()), len(ts)],
    marker=dict(color=[MUTED, BLUE, BLUE, AQUA]), texttemplate="%{value:,}<br>%{percentInitial:.0%} of raw",
    connector=dict(fillcolor="#f0efec"),
    hovertemplate="%{y}: %{x:,} rows<extra></extra>",
))
fig.update_layout(title="Figure 7. Cleaning pipeline: from raw rows to one row per hour", height=380,
                  margin=dict(l=230))
fig.show()

In [32]:
gaps_win = gap_table(is_missing[WINDOW_START:WINDOW_END])
g = gaps_win.iloc[0]
lo, hi = g.start - pd.Timedelta(hours=36), g.end + pd.Timedelta(hours=36)
seg = ts.loc[lo:hi]
imp = seg[seg["is_imputed"] == 1]

fig = go.Figure()
fig.add_vrect(x0=g.start - pd.Timedelta(minutes=30), x1=g.end + pd.Timedelta(minutes=30),
              fillcolor=ORANGE, opacity=0.08, line_width=0)
fig.add_trace(go.Scatter(x=seg.index, y=seg["traffic_volume"], mode="lines", line=dict(color=BLUE, width=2),
                         name="Cleaned series", hovertemplate="%{x|%d/%m %H:%M}: %{y:,.0f} vehicles<extra></extra>"))
fig.add_trace(go.Scatter(x=imp.index, y=imp["traffic_volume"], mode="markers",
                         marker=dict(color=ORANGE, size=9, line=dict(color="#ffffff", width=2)),
                         name="Interpolated hours", hovertemplate="%{x|%d/%m %H:%M}: %{y:,.0f} vehicles (interpolated)<extra></extra>"))
fig.update_layout(title=f"Figure 8. Longest gap in 2016–2018 ({g.hours} hours) after time-based interpolation",
                  yaxis_title="Vehicles/hour", height=400)
fig.show()

**Nhận xét.** Các giờ nội suy (cam) nối liền mạch hai đầu đoạn thiếu. Vì gap dài nhất chỉ 9 giờ nên nội suy tuyến tính không bịa ra cao điểm giả; tuy vậy cột `is_imputed` vẫn được giữ để ở phần mô hình hóa có thể **chỉ đánh giá trên giờ thật**.

<a id="5"></a>
## 5. EDA trực quan

Từ đây mọi phân tích dùng chuỗi đã làm sạch `ts` (01/2016 – 09/2018, mỗi giờ một dòng). Mỗi mục đặt ra một câu hỏi, trả lời bằng biểu đồ và rút ra hệ quả cho bước mô hình hóa.

### 5.1 Toàn cảnh chuỗi

**Câu hỏi:** chuỗi có xu hướng tăng/giảm dài hạn không, và các mẫu lặp lại nằm ở thang thời gian nào?

In [33]:
daily = y.resample("D").mean()
fig = go.Figure()
fig.add_trace(go.Scatter(x=daily.index, y=daily.values, mode="lines", line=dict(color=BLUE, width=1),
                         name="Daily mean", opacity=0.75, hovertemplate="%{x|%a %d/%m/%Y}: %{y:,.0f}<extra></extra>"))
fig.add_trace(go.Scatter(x=daily.index, y=daily.rolling(28, center=True).mean().values, mode="lines",
                         line=dict(color=ORANGE, width=3), name="28-day rolling mean",
                         hovertemplate="%{x|%d/%m/%Y}: %{y:,.0f}<extra></extra>"))
fig.update_layout(title="Figure 9. Daily mean traffic volume 2016–2018: no clear long-term upward or downward trend",
                  yaxis_title="Vehicles/hour (daily mean)", height=430, hovermode="x unified",
                  xaxis=dict(rangeslider=dict(visible=True, thickness=0.06)))
fig.show()

Biểu đồ đường chỉ cho thấy trung bình ngày. Để thấy **toàn bộ 24.096 giờ cùng lúc**, xếp mỗi ngày thành một cột và mỗi giờ thành một hàng — một "dấu vân tay" của chuỗi:

In [34]:
fp = y.to_frame("v").assign(date=y.index.normalize(), hour=y.index.hour).pivot(index="hour", columns="date", values="v")
fig = go.Figure(go.Heatmap(z=fp.values, x=fp.columns, y=fp.index, colorscale=SEQ_BLUE, zmin=0, zmax=7000,
                           colorbar=dict(title="Vehicles<br>/hour"),
                           hovertemplate="%{x|%a %d/%m/%Y} %{y}h: %{z:,.0f}<extra></extra>"))
fig.update_layout(title="Figure 10. Every hour of 2016–2018: rows = hour of day, columns = days. "
                        "Weekly stripes, weekday rush hours, holidays & closures as gaps",
                  yaxis=dict(title="Hour of day", dtick=3, autorange="reversed"), height=460)
fig.show()

**Đọc biểu đồ.** Mỗi điểm ảnh là một giờ. Hai dải đậm ngang ở khoảng 6–8h và 15–17h là **cao điểm sáng/chiều**; chúng bị cắt đều đặn bởi các **sọc dọc nhạt** — chính là thứ Bảy và Chủ nhật (chu kỳ tuần). Các vạch trắng bất thường giữa tuần (vd. cuối 07/2016, đầu 2018) là ngày lễ, ngày đường bị đóng hoặc bão tuyết. Không có vùng nào sáng dần/tối dần theo thời gian → **không có xu hướng dài hạn**.

Lịch dưới đây gom về tổng lưu lượng **mỗi ngày**, xếp theo tuần × thứ cho từng năm:

In [35]:
day_total = y.resample("D").sum()
cal_df = day_total.to_frame("total").assign(year=day_total.index.year, dow=day_total.index.dayofweek)
jan1_dow = pd.to_datetime(cal_df["year"].astype(str) + "-01-01").dt.dayofweek.to_numpy()
cal_df["week"] = (day_total.index.dayofyear - 1 + jan1_dow) // 7
years = sorted(cal_df["year"].unique())

fig = make_subplots(rows=len(years), cols=1, vertical_spacing=0.06, subplot_titles=[str(yr) for yr in years])
for i, yr in enumerate(years, start=1):
    part = cal_df[cal_df["year"] == yr]
    grid = part.pivot(index="dow", columns="week", values="total").reindex(index=range(7))
    dates = part.assign(d=part.index.strftime("%a %d/%m/%Y")).pivot(index="dow", columns="week", values="d").reindex(index=range(7))
    fig.add_trace(go.Heatmap(z=grid.values, x=grid.columns, y=DAY_LABELS, customdata=dates.values,
                             colorscale=SEQ_BLUE, zmin=day_total.quantile(0.01), zmax=day_total.quantile(0.99),
                             xgap=2, ygap=2, showscale=(i == 1), colorbar=dict(title="Vehicles<br>/day", len=0.9),
                             hovertemplate="%{customdata}: %{z:,.0f} vehicles<extra></extra>"), row=i, col=1)
    fig.update_yaxes(autorange="reversed", row=i, col=1)
    fig.update_xaxes(range=[-0.5, 53.5], showgrid=False, row=i, col=1)
fig.update_xaxes(title_text="Week of year", row=len(years), col=1)
fig.update_layout(title="Figure 11. Calendar heatmap of daily totals: weekends are always pale, holidays stand out as pale weekdays",
                  height=560, plot_bgcolor="#ffffff")
fig.show()

**Nhận xét.** Hai hàng dưới cùng (Sat, Sun) luôn nhạt hơn — chu kỳ tuần rất đều. Các ô nhạt lẻ loi giữa tuần rơi đúng vào ngày lễ (Memorial Day cuối tháng 5, Independence Day 4/7, Thanksgiving cuối tháng 11, Giáng sinh – Năm mới) và vài ngày bão tuyết mùa đông. Phóng to vào **hai tuần bất kỳ** để thấy hình dạng trong ngày:

In [36]:
week2 = ts.loc["2017-10-02":"2017-10-15"]
fig = go.Figure(go.Scatter(x=week2.index, y=week2["traffic_volume"], mode="lines", line=dict(color=BLUE, width=2),
                           fill="tozeroy", fillcolor="rgba(42,120,214,0.10)",
                           hovertemplate="%{x|%a %d/%m %H:%M}: %{y:,.0f} vehicles<extra></extra>"))
for d in pd.date_range("2017-10-07", "2017-10-15", freq="7D"):
    fig.add_vrect(x0=d, x1=d + pd.Timedelta(days=2), fillcolor=MUTED, opacity=0.12, line_width=0,
                  annotation_text="weekend", annotation_position="top left", annotation_font_color=INK_2)
fig.update_layout(title="Figure 12. Two weeks, 2–15 Oct 2017: two peaks on weekdays, one lower and flatter peak on weekends",
                  yaxis_title="Vehicles/hour", height=400, showlegend=False)
fig.show()

**Kết luận 5.1.** Chuỗi **không có xu hướng dài hạn** rõ rệt; biến động chủ yếu đến từ chu kỳ **ngày** (2 cao điểm vào ngày thường) và chu kỳ **tuần** (cuối tuần thấp và chỉ có 1 đỉnh). Các ngày lễ và sự kiện đặc biệt tạo ra những "lỗ" giữa tuần.

### 5.2 Phân phối của biến mục tiêu

In [37]:
vals = y.to_numpy()
fig = go.Figure(hist_bar(vals, bins=np.arange(0, 7600, 200), color=BLUE))
fig.add_vrect(x0=0, x1=1000, fillcolor=MUTED, opacity=0.08, line_width=0, annotation_text="night",
              annotation_position="top left", annotation_font_color=INK_2)
fig.add_vrect(x0=4000, x1=7400, fillcolor=BLUE, opacity=0.06, line_width=0, annotation_text="daytime",
              annotation_position="top right", annotation_font_color=BLUE)
fig.add_vline(x=np.median(vals), line_dash="dot", line_color=INK_2,
              annotation_text=f"Median {np.median(vals):,.0f}", annotation_position="top left",
              annotation_font_color=INK_2)
fig.update_layout(title="Figure 13. traffic_volume is bimodal: night hours (< 1,000) and daytime hours (4,000–6,000)",
                  xaxis_title="Vehicles/hour", yaxis_title="Hours", height=400, showlegend=False)
fig.show()

In [38]:
print(f"Tỷ lệ giờ < 1.000 xe: {(vals < 1000).mean():.1%} | 1.000–4.000: {((vals >= 1000) & (vals < 4000)).mean():.1%} "
      f"| ≥ 4.000: {(vals >= 4000).mean():.1%}")
print(f"Trung bình: {vals.mean():,.0f} | Độ lệch chuẩn: {vals.std():,.0f} xe/giờ (≈ {vals.std() / vals.mean():.0%} trung bình)")

Tỷ lệ giờ < 1.000 xe: 21.6% | 1.000–4.000: 33.5% | ≥ 4.000: 44.9%
Trung bình: 3,313 | Độ lệch chuẩn: 1,963 xe/giờ (≈ 59% trung bình)


Hai đỉnh của histogram thực chất là **hai nhóm giờ khác nhau**. Tách phân phối theo từng giờ trong ngày (dải đậm = 50% giờ ở giữa, dải nhạt = 90%, đường = trung vị):

In [39]:
is_wkend = np.where(y.index.dayofweek >= 5, "Weekend", "Weekday (Mon–Fri)")
qs = (y.groupby([is_wkend, y.index.hour]).quantile([0.05, 0.25, 0.5, 0.75, 0.95]).unstack())
hours = np.arange(24)

fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.04,
                    subplot_titles=("Weekday (Mon–Fri)", "Weekend"))
for c, (name, color, rgba) in enumerate([("Weekday (Mon–Fri)", BLUE, "42,120,214"), ("Weekend", ORANGE, "235,104,52")], start=1):
    q = qs.loc[name]
    band(fig, hours, q[0.05].values, q[0.95].values, f"rgba({rgba},0.15)", "5–95%", row=1, col=c)
    band(fig, hours, q[0.25].values, q[0.75].values, f"rgba({rgba},0.35)", "25–75%", row=1, col=c)
    fig.add_trace(go.Scatter(x=hours, y=q[0.5].values, mode="lines+markers", line=dict(color=color, width=2.5),
                             marker=dict(size=6), name="Median",
                             hovertemplate="%{x}h: median %{y:,.0f}<extra>" + name + "</extra>"), row=1, col=c)
fig.update_xaxes(title_text="Hour of day", dtick=3)
fig.update_yaxes(title_text="Vehicles/hour", col=1)
fig.update_layout(title="Figure 14. Distribution of traffic by hour: weekday rush hours are high and tight, weekend traffic peaks around noon",
                  height=440, showlegend=False)
fig.show()

**Nhận xét.** Phân phối **không chuẩn, có hai đỉnh**: khoảng 22% số giờ dưới 1.000 xe (đêm khuya) và 45% số giờ từ 4.000 xe trở lên (ban ngày). Độ lệch chuẩn bằng gần 60% trung bình — nhưng Figure 14 cho thấy trong **từng giờ cụ thể**, dải 25–75% khá hẹp: phần lớn biến động là do giờ trong ngày chứ không phải nhiễu. → Mô hình **bắt buộc** phải có thông tin giờ (đặc trưng lịch hoặc `lag_24`). Ngoài ra, dải 5–95% của ngày thường kéo xuống rất thấp vào giờ cao điểm — đó là các ngày lễ/đóng đường lẫn vào, sẽ xử lý ở 5.4 và 5.9.

### 5.3 Mùa vụ theo giờ trong ngày, thứ trong tuần và tháng

In [40]:
cal = pd.DataFrame({"y": y, "hour": y.index.hour, "dow": y.index.dayofweek, "month": y.index.month})
pivot = cal.pivot_table(index="dow", columns="hour", values="y", aggfunc="mean")

fig = go.Figure(go.Heatmap(z=pivot.values, x=[f"{h:02d}h" for h in pivot.columns], y=DAY_LABELS,
                           colorscale=SEQ_BLUE, colorbar=dict(title="Vehicles/hour"), xgap=2, ygap=2,
                           text=(pivot.values / 1000).round(1), texttemplate="%{text}", textfont=dict(size=9),
                           hovertemplate="%{y} %{x}: %{z:,.0f} vehicles/hour<extra></extra>"))
fig.update_layout(title="Figure 15. Mean traffic volume (thousands) by hour × weekday: 7am and 4pm peaks occur only Monday–Friday",
                  yaxis_autorange="reversed", height=420)
fig.show()

Cùng dữ liệu nhưng vẽ trên **mặt đồng hồ 24 giờ** — mỗi đường là một thứ; hình dạng "hai cánh" của ngày thường và "một cánh" của cuối tuần hiện ra rất rõ:

In [41]:
theta = [f"{h:02d}h" for h in range(24)]
fig = go.Figure()
day_colors = [BLUE] * 5 + [ORANGE, RED]
for d in range(7):
    r = pivot.loc[d].values
    fig.add_trace(go.Scatterpolar(r=np.r_[r, r[0]], theta=theta + theta[:1], mode="lines", name=DAY_LABELS[d],
                                  line=dict(color=day_colors[d], width=2.5 if d >= 5 else 1.5),
                                  opacity=1 if d >= 5 else 0.7,
                                  hovertemplate="%{theta}: %{r:,.0f}<extra>" + DAY_LABELS[d] + "</extra>"))
fig.update_layout(title="Figure 16. 24-hour clock of mean traffic: weekdays (blue) have morning + afternoon lobes, Sat/Sun one midday lobe",
                  height=560, polar=dict(angularaxis=dict(direction="clockwise", rotation=90, gridcolor=GRID),
                                         radialaxis=dict(gridcolor=GRID, tickvals=[2000, 4000, 6000], ticktext=["2k", "4k", "6k"],
                                                         angle=45, tickangle=45, tickfont=dict(size=10, color=INK_2))),
                  legend=dict(orientation="v", x=1.02, y=0.5, yanchor="middle"))
fig.show()

In [42]:
mth = pd.DataFrame({"total": day_total, "month": day_total.index.month,
                    "type": np.where(day_total.index.dayofweek >= 5, "Weekend", "Weekday (Mon–Fri)")})
fig = go.Figure()
for name, color in [("Weekday (Mon–Fri)", BLUE), ("Weekend", ORANGE)]:
    part = mth[mth["type"] == name]
    fig.add_trace(box_from_groups(part["total"], part["month"].map(dict(enumerate(MONTH_LABELS, start=1))),
                                  color=color, name=name))
fig.update_xaxes(categoryorder="array", categoryarray=MONTH_LABELS)
fig.update_layout(title="Figure 17. Daily total by month (box = 25–75%, whiskers = 5–95%): yearly seasonality is mild",
                  yaxis_title="Vehicles/day (total)", height=430, boxmode="group")
fig.show()

In [43]:
monthly = cal.groupby("month")["y"].mean()
hour_mean = cal.groupby("hour")["y"].mean()
dow_mean = cal.groupby("dow")["y"].mean()
pd.DataFrame({
    "Thang thời gian": ["Giờ trong ngày", "Thứ trong tuần", "Tháng trong năm"],
    "Cao nhất": [f"{hour_mean.idxmax()}h ({hour_mean.max():,.0f})", f"{DAY_LABELS[dow_mean.idxmax()]} ({dow_mean.max():,.0f})",
                 f"{MONTH_LABELS[monthly.idxmax() - 1]} ({monthly.max():,.0f})"],
    "Thấp nhất": [f"{hour_mean.idxmin()}h ({hour_mean.min():,.0f})", f"{DAY_LABELS[dow_mean.idxmin()]} ({dow_mean.min():,.0f})",
                  f"{MONTH_LABELS[monthly.idxmin() - 1]} ({monthly.min():,.0f})"],
    "Cao / thấp": [hour_mean.max() / hour_mean.min(), dow_mean.max() / dow_mean.min(), monthly.max() / monthly.min()],
})

,Thang thời gian,Cao nhất,Thấp nhất,Cao / thấp
0,Giờ trong ngày,"16h (5,725)",3h (395),14.49
1,Thứ trong tuần,"Fri (3,701)","Sun (2,471)",1.50
2,Tháng trong năm,"Mar (3,472)","Dec (3,106)",1.12


Mùa trong năm có làm **thay đổi hình dạng** của ngày không (ví dụ mùa đông cao điểm thấp hơn)? So sánh hồ sơ giờ của ngày thường theo bốn mùa:

In [44]:
season_of = {12: "Winter (Dec–Feb)", 1: "Winter (Dec–Feb)", 2: "Winter (Dec–Feb)", 3: "Spring (Mar–May)",
             4: "Spring (Mar–May)", 5: "Spring (Mar–May)", 6: "Summer (Jun–Aug)", 7: "Summer (Jun–Aug)",
             8: "Summer (Jun–Aug)", 9: "Fall (Sep–Nov)", 10: "Fall (Sep–Nov)", 11: "Fall (Sep–Nov)"}
wk = ts[(ts.index.dayofweek < 5) & (ts["is_holiday"] == 0)]
season_prof = wk.groupby([wk.index.month.map(season_of), wk.index.hour])["traffic_volume"].mean().unstack(0)

fig = go.Figure()
for name, color in [("Winter (Dec–Feb)", BLUE), ("Spring (Mar–May)", AQUA), ("Summer (Jun–Aug)", ORANGE),
                    ("Fall (Sep–Nov)", YELLOW)]:
    fig.add_trace(go.Scatter(x=season_prof.index, y=season_prof[name], mode="lines", name=name,
                             line=dict(color=color, width=2.5),
                             hovertemplate="%{x}h: %{y:,.0f}<extra>" + name + "</extra>"))
fig.update_layout(title="Figure 18. Weekday hourly profile by season: same shape all year, the winter morning peak is ~6% lower",
                  xaxis=dict(title="Hour of day", dtick=2), yaxis_title="Vehicles/hour (mean)", height=420,
                  hovermode="x unified")
fig.show()

**Nhận xét.** Giờ cao nhất (16h) đông gấp **khoảng 14,5 lần** giờ thấp nhất (3h), trong khi tháng cao nhất chỉ hơn tháng thấp nhất khoảng **12%**. Heatmap và mặt đồng hồ cho thấy hai cao điểm 6–7h và 15–17h chỉ xuất hiện từ T2 đến T6 — đặc trưng của lưu lượng đi làm; cuối tuần chỉ có một đỉnh rộng quanh trưa. Hồ sơ giờ gần như **giữ nguyên hình dạng qua bốn mùa** (Figure 18); khác biệt rõ nhất là cao điểm sáng mùa đông thấp hơn khoảng 6%. → Chu kỳ **ngày** và **tuần** là quan trọng nhất; chu kỳ năm chỉ là hiệu chỉnh nhỏ (đặc trưng `month`), và tương tác *giờ × thứ* quan trọng hơn tương tác *giờ × mùa*.

### 5.4 Câu hỏi: ngày lễ làm lưu lượng giảm bao nhiêu?

In [45]:
day_type = np.select([ts["is_holiday"] == 1, ts.index.dayofweek >= 5], ["Holiday", "Weekend"], "Weekday (Mon–Fri)")
order = ["Weekday (Mon–Fri)", "Weekend", "Holiday"]
prof = (pd.DataFrame({"y": y, "hour": y.index.hour, "type": day_type})
          .groupby(["type", "hour"])["y"].mean().unstack(0))
daily_by_type = (pd.DataFrame({"y": y, "date": y.index.normalize(), "type": day_type})
                   .groupby(["type", "date"])["y"].sum().groupby("type").mean().reindex(order))
holiday_drop = 1 - daily_by_type["Holiday"] / daily_by_type["Weekday (Mon–Fri)"]
weekend_drop = 1 - daily_by_type["Weekend"] / daily_by_type["Weekday (Mon–Fri)"]

In [46]:
fig = go.Figure()
band(fig, prof.index.to_numpy(), prof["Holiday"].values, prof["Weekday (Mon–Fri)"].values,
     "rgba(27,175,122,0.12)", "Traffic lost on holidays")
for name, color in zip(order, [BLUE, ORANGE, AQUA]):
    fig.add_trace(go.Scatter(x=prof.index, y=prof[name], mode="lines+markers", name=name,
                             line=dict(color=color, width=2.5), marker=dict(size=7, line=dict(color="#ffffff", width=1.5)),
                             hovertemplate="%{y:,.0f}<extra>" + name + "</extra>"))
fig.update_layout(title=f"Figure 19. Hourly profile: holidays carry {holiday_drop:.0%} less daily traffic than weekdays, "
                        f"weekends {weekend_drop:.0%} less",
                  xaxis=dict(title="Hour of day", dtick=2), yaxis_title="Vehicles/hour (mean)", height=430,
                  hovermode="x unified")
fig.show()

In [47]:
daily_by_type.rename("Tổng xe/ngày (trung bình)").to_frame().assign(
    **{"So với ngày thường": lambda d: (d.iloc[:, 0] / d.iloc[0, 0] - 1).map("{:+.1%}".format)})

,Tổng xe/ngày (trung bình),So với ngày thường
type,,
Weekday (Mon–Fri),"86,656.97",+0.0%
Weekend,"64,092.21",-26.0%
Holiday,"62,317.48",-28.1%


Trung bình gộp có thể che mất khác biệt giữa các ngày lễ. Với **từng ngày lễ**, so tổng lưu lượng thực tế với "ngày bình thường kỳ vọng" = trung vị tổng lưu lượng của cùng thứ trong 4 tuần trước và 4 tuần sau (không tính ngày lễ):

In [48]:
is_hol_day = ts["is_holiday"].resample("D").max().astype(bool)
holiday_name = ts["holiday"].resample("D").first()
rows = []
for d in day_total.index[is_hol_day]:
    same_dow = [d + pd.Timedelta(weeks=k) for k in range(-4, 5) if k != 0]
    same_dow = [x for x in same_dow if x in day_total.index and not is_hol_day[x]]
    rows.append({"date": d, "holiday": holiday_name[d], "actual": day_total[d],
                 "expected": day_total[same_dow].median()})
hol = pd.DataFrame(rows).assign(drop=lambda d: d["actual"] / d["expected"] - 1).sort_values("drop")
hol["label"] = hol["holiday"] + " · " + hol["date"].dt.strftime("%a %d/%m/%y")

fig = go.Figure()
for _, r in hol.iterrows():
    fig.add_shape(type="line", x0=r["actual"], x1=r["expected"], y0=r["label"], y1=r["label"],
                  line=dict(color=GRID, width=4))
fig.add_trace(go.Scatter(x=hol["expected"], y=hol["label"], mode="markers", name="Expected (same weekday, ±4 weeks)",
                         marker=dict(color=MUTED, size=10), hovertemplate="expected %{x:,.0f}<extra></extra>"))
fig.add_trace(go.Scatter(x=hol["actual"], y=hol["label"], mode="markers+text", name="Actual on the holiday",
                         marker=dict(color=AQUA, size=12, line=dict(color="#ffffff", width=1.5)),
                         text=[f"{v:+.0%}" for v in hol["drop"]], textfont_color=INK_2,
                         textposition=np.where(hol["drop"] < 0, "middle left", "middle right"),
                         hovertemplate="actual %{x:,.0f}<extra></extra>"))
fig.update_layout(title="Figure 20. Each holiday vs. an ordinary day of the same weekday: from −49% (New Year) to +5% (State Fair)",
                  xaxis=dict(title="Vehicles/day (total)", range=[0, hol["expected"].max() * 1.08]),
                  yaxis=dict(autorange="reversed", tickfont=dict(size=11)), height=760, margin=dict(l=280))
fig.show()

In [49]:
hol_by_name = hol.groupby("holiday")["drop"].agg(["mean", "min", "max", "size"]).sort_values("mean")
hol_by_name.columns = ["Giảm trung bình", "Giảm nhiều nhất", "Giảm ít nhất", "Số lần"]
hol_by_name.style.format({c: "{:+.1%}" for c in hol_by_name.columns[:3]})

,Giảm trung bình,Giảm nhiều nhất,Giảm ít nhất,Số lần
holiday,,,,
Thanksgiving Day,-47.7%,-49.0%,-46.4%,2
New Years Day,-44.1%,-49.0%,-35.5%,3
Independence Day,-43.7%,-48.4%,-40.8%,3
Christmas Day,-39.0%,-45.0%,-33.1%,2
Memorial Day,-38.3%,-40.0%,-37.4%,3
Labor Day,-33.7%,-34.6%,-33.1%,3
Martin Luther King Jr Day,-17.7%,-21.5%,-13.8%,2
Washingtons Birthday,-15.8%,-28.3%,-8.8%,3
Veterans Day,-3.1%,-3.8%,-2.4%,2


**Kết luận 5.4.** Tính trung bình, tổng lưu lượng một **ngày lễ thấp hơn ngày thường khoảng 28%**, gần bằng mức giảm của cuối tuần (26%), và cao điểm sáng của ngày lễ chỉ còn khoảng một nửa ngày thường. Nhưng Figure 20 cho thấy **mức giảm rất khác nhau giữa các ngày lễ**:
- Lễ lớn (Năm mới, Thanksgiving, Independence Day, Giáng sinh, Memorial Day, Labor Day): giảm **33–49%**.
- Martin Luther King Jr Day, Washington's Birthday: giảm 9–28% (nhiều nơi vẫn làm việc).
- Veterans Day, Columbus Day: gần như không giảm (−4% đến 0%).
- **State Fair** (hội chợ bang Minnesota, không phải ngày nghỉ) còn làm lưu lượng **tăng** 1–5%.

→ Biến `is_holiday` là cần thiết, nhưng một cờ nhị phân sẽ gộp chung các trường hợp trái ngược nhau; nên phân biệt **lễ lớn / lễ nhỏ** (hoặc bỏ State Fair khỏi nhóm "ngày lễ").

### 5.5 Câu hỏi: thời tiết xấu làm lưu lượng giảm bao nhiêu?

So sánh thẳng trung bình theo `weather_main` sẽ bị nhiễu: ví dụ sương mù (`Fog`, `Mist`) hay xảy ra lúc rạng sáng — giờ vốn vắng xe. Để công bằng, mỗi giờ được so với **trung vị của chính khung giờ đó** (cùng thứ, cùng giờ, ngày không lễ):

$$\text{chênh lệch} = \frac{y_t - \text{median}(y \mid \text{thứ}, \text{giờ})}{\text{median}(y \mid \text{thứ}, \text{giờ})}$$

Chỉ dùng giờ ban ngày 6h–20h (mẫu số đủ lớn) và giờ có dữ liệu thật (không nội suy).

In [50]:
base = ts[(ts["is_holiday"] == 0) & (ts["is_imputed"] == 0)].copy()
base["how"] = base.index.dayofweek * 24 + base.index.hour
base["expected"] = base.groupby("how")["traffic_volume"].transform("median")
day = base[(base.index.hour >= 6) & (base.index.hour <= 20)]
day = day.assign(rel=(day["traffic_volume"] / day["expected"] - 1) * 100)
print(f"Số giờ ban ngày dùng để so sánh: {len(day):,}")

Số giờ ban ngày dùng để so sánh: 13,948


In [51]:
def effect_table(groups, min_n=30):
    t = day.groupby(groups)["rel"].agg(["mean", "std", "count"]).rename(columns={"count": "n"})
    t = t[t["n"] >= min_n]
    t["ci95"] = 1.96 * t["std"] / np.sqrt(t["n"])
    return t.sort_values("mean")


eff = effect_table(day["weather_main"])
eff

,mean,std,n,ci95
weather_main,,,,
Snow,-10.63,17.30,879,1.14
Thunderstorm,-3.04,10.33,210,1.40
Mist,-1.87,11.21,907,0.73
Rain,-1.81,11.00,1645,0.53
Haze,-0.67,9.69,290,1.12
Clear,-0.42,9.70,4172,0.29
Drizzle,-0.26,7.81,196,1.09
Clouds,-0.17,9.92,5577,0.26
Fog,-0.02,7.38,66,1.78


In [52]:
fig = go.Figure(go.Bar(x=eff["mean"], y=eff.index, orientation="h",
                       marker_color=np.where(eff["mean"] < -5, RED, np.where(eff["mean"] < 0, "#f2a3a2", BLUE)),
                       error_x=dict(type="data", array=eff["ci95"], color=INK_2, thickness=1.2, width=4),
                       hovertemplate="%{y}: %{x:+.1f}% vs. same weekday-hour<extra></extra>"))
fig.add_trace(go.Scatter(x=np.minimum(eff["mean"], 0) - eff["ci95"] - 0.3, y=eff.index, mode="text",
                         text=[f"{m:+.1f}%  (n={n:,})" for m, n in zip(eff["mean"], eff["n"])],
                         textposition="middle left", textfont_color=INK_2, hoverinfo="skip"))
fig.add_vline(x=0, line_color=INK_2, line_width=1)
fig.update_layout(title="Figure 21. Traffic deviation from the same weekday-hour baseline, by weather type (6am–8pm, ±95% CI)",
                  xaxis_title="Mean deviation (%)", height=430, showlegend=False,
                  xaxis_range=[eff["mean"].min() * 1.6 - 1, 3])
fig.show()

`weather_main` vẫn còn thô: "Snow" gồm cả tuyết nhẹ và bão tuyết. Tách theo `weather_description` cho mưa và tuyết:

In [53]:
rs = day[day["weather_main"].isin(["Rain", "Snow", "Drizzle", "Thunderstorm"])]
eff_desc = effect_table(rs["weather_description"], min_n=20)
grp = rs.groupby("weather_description")["weather_main"].first()
color_map = {"Rain": BLUE, "Snow": RED, "Drizzle": AQUA, "Thunderstorm": YELLOW}

fig = go.Figure()
for g_name, color in color_map.items():
    part = eff_desc[grp.reindex(eff_desc.index) == g_name]
    if part.empty:
        continue
    fig.add_trace(go.Scatter(x=part["mean"], y=part.index, mode="markers", name=g_name,
                             marker=dict(color=color, size=np.clip(np.sqrt(part["n"]) * 1.4, 7, 30),
                                         line=dict(color="#ffffff", width=1.5)),
                             error_x=dict(type="data", array=part["ci95"], color=color, thickness=1.2, width=0),
                             customdata=part["n"],
                             hovertemplate="%{y}: %{x:+.1f}% (n=%{customdata:,})<extra></extra>"))
fig.add_vline(x=0, line_color=INK_2, line_width=1)
fig.update_layout(title="Figure 22. Precipitation intensity matters: heavy snow cuts traffic far more than light snow (dot size ∝ √n)",
                  xaxis_title="Mean deviation vs. same weekday-hour (%)", height=520,
                  yaxis=dict(categoryorder="array", categoryarray=list(eff_desc.index)))
fig.show()

In [54]:
snow_rel = day.loc[day["weather_main"] == "Snow", "rel"]
heavy = day["weather_description"].str.contains("heavy", case=False)
print(f"Giờ có tuyết: {snow_rel.mean():+.1f}% (n={len(snow_rel):,})")
print(f"Giờ có hiện tượng 'heavy' (mưa/tuyết lớn): {day.loc[heavy, 'rel'].mean():+.1f}% (n={heavy.sum():,})")
print(f"Giờ trời quang (Clear): {day.loc[day['weather_main'] == 'Clear', 'rel'].mean():+.1f}%")

Giờ có tuyết: -10.6% (n=879)
Giờ có hiện tượng 'heavy' (mưa/tuyết lớn): -13.6% (n=201)
Giờ trời quang (Clear): -0.4%


In [55]:
dd = pd.DataFrame({"total": day_total, "temp": ts["temp_c"].resample("D").mean()})
dd["type"] = np.where(dd.index.dayofweek >= 5, "Weekend", "Weekday (Mon–Fri)")
dd = dd[~is_hol_day]

fig = go.Figure()
for name, color in [("Weekday (Mon–Fri)", BLUE), ("Weekend", ORANGE)]:
    part = dd[dd["type"] == name]
    fig.add_trace(go.Scatter(x=part["temp"], y=part["total"], mode="markers", name=name,
                             marker=dict(color=color, size=7, opacity=0.6, line=dict(color="#ffffff", width=0.8)),
                             hovertemplate="%{x:.1f} °C: %{y:,.0f} vehicles/day<extra></extra>"))
    # Đường xu hướng: trung vị theo từng khoảng 5 °C
    bins_t = pd.cut(part["temp"], np.arange(-30, 40, 5))
    trend = part.groupby(bins_t, observed=True)["total"].median()
    fig.add_trace(go.Scatter(x=[b.mid for b in trend.index], y=trend.values, mode="lines", showlegend=False,
                             line=dict(color=color, width=3), hoverinfo="skip"))
corr_wd = dd.loc[dd["type"] == "Weekday (Mon–Fri)", ["temp", "total"]].corr().iloc[0, 1]
fig.update_layout(title=f"Figure 23. Daily temperature vs. daily total (holidays excluded): weak link, r = {corr_wd:.2f} on weekdays; "
                        "very cold days dip",
                  xaxis_title="Daily mean temperature (°C)", yaxis_title="Vehicles/day (total)", height=440)
fig.show()

**Kết luận 5.5.** Sau khi khử ảnh hưởng của giờ và thứ, phần lớn loại thời tiết chỉ làm lưu lượng lệch dưới 2–3% — gần như không đáng kể. Ngoại lệ rõ rệt là **tuyết: −10,6%** và các giờ mưa/tuyết **"heavy": −13,6%**; Figure 22 cho thấy mức giảm **tăng theo cường độ**: tuyết nhẹ ≈ −7%, tuyết vừa ≈ −27%, **tuyết lớn ≈ −43%**; mưa lớn ≈ −8% trong khi mưa nhẹ/vừa gần như không ảnh hưởng. Nhiệt độ hầu như không liên quan đến tổng lưu lượng (r = 0,14), chỉ những ngày rét dưới khoảng −15 °C mới giảm rõ (thường trùng bão tuyết). → Thời tiết chỉ quan trọng ở các tình huống cực đoan và tác động **phi tuyến**; kỳ vọng các đặc trưng thời tiết đóng góp ít cho mô hình tuyến tính, mô hình cây có thể khai thác tốt hơn.

### 5.6 Phân rã mùa vụ (MSTL)

**Ý tưởng.** Một chuỗi thời gian có thể xem như **tổng của các "lớp"** chồng lên nhau:

$$y_t = \underbrace{T_t}_{\text{xu hướng}} + \underbrace{S^{(24)}_t}_{\text{mùa vụ ngày}} + \underbrace{S^{(168)}_t}_{\text{mùa vụ tuần}} + \underbrace{R_t}_{\text{phần dư}}$$

- **Xu hướng $T_t$**: mức nền thay đổi chậm qua nhiều tuần/tháng (đường tăng/giảm dài hạn).
- **Mùa vụ ngày $S^{(24)}_t$**: dạng sóng lặp lại mỗi 24 giờ (cao điểm sáng/chiều, đêm vắng).
- **Mùa vụ tuần $S^{(168)}_t$**: phần chênh thêm lặp lại mỗi 168 giờ = 7 ngày (ngày thường cao, cuối tuần thấp).
- **Phần dư $R_t$**: những gì còn lại — sự kiện bất thường, thời tiết, nhiễu.

**MSTL** (*Multiple Seasonal-Trend decomposition using LOESS*) tách lần lượt từng lớp: làm trơn để lấy mùa vụ 24h, trừ đi, rồi lấy mùa vụ 168h, lặp lại vài vòng cho ổn định; phần trơn còn lại là xu hướng. Khác với STL thường (chỉ một chu kỳ), MSTL xử lý được **hai chu kỳ lồng nhau** như chuỗi này.

**Đo "lớp nào quan trọng"** bằng hai chỉ số:
- **% phương sai**: $\text{Var}(\text{thành phần}) / \text{Var}(y)$ — thành phần đó giải thích bao nhiêu phần dao động của chuỗi.
- **Độ mạnh mùa vụ** (Wang, Smith & Hyndman 2006): $F_S = \max\left(0,\ 1 - \dfrac{\text{Var}(R)}{\text{Var}(S + R)}\right)$ — gần 1 nghĩa là mùa vụ lấn át hẳn nhiễu.

In [56]:
mstl = MSTL(y, periods=(24, 168), windows=(25, 169)).fit()
comp = pd.DataFrame({"Observed": y, "Trend": mstl.trend, "Daily seasonality (24h)": mstl.seasonal["seasonal_24"],
                     "Weekly seasonality (168h)": mstl.seasonal["seasonal_168"], "Residual": mstl.resid})
comp.head()

,Observed,Trend,Daily seasonality (24h),Weekly seasonality (168h),Residual
date_time,,,,,
2016-01-01 00:00:00,"1,513.00","2,345.45","-1,857.12",-57.06,"1,081.73"
2016-01-01 01:00:00,"1,550.00","2,353.73","-2,026.50",-33.79,"1,256.56"
2016-01-01 02:00:00,"1,134.50","2,361.98","-2,147.88",-21.90,942.30
2016-01-01 03:00:00,719.00,"2,370.20","-2,211.68",-5.82,566.30
2016-01-01 04:00:00,533.00,"2,378.38","-1,872.95",118.96,-91.39


In [57]:
share = comp.drop(columns="Observed").var() / comp["Observed"].var() * 100
strength = pd.Series({k: max(0, 1 - mstl.resid.var() / (mstl.resid + comp[k]).var())
                      for k in ["Daily seasonality (24h)", "Weekly seasonality (168h)"]})
pd.DataFrame({"% phương sai của chuỗi": share, "Độ mạnh mùa vụ (0–1)": strength})

,% phương sai của chuỗi,Độ mạnh mùa vụ (0–1)
Daily seasonality (24h),79.04,0.95
Residual,3.89,NaN
Trend,1.15,NaN
Weekly seasonality (168h),15.36,0.80


In [58]:
sl = comp.loc["2018-03-05":"2018-03-25"]
colors = [BLUE, ORANGE, AQUA, YELLOW, MUTED]
fig = make_subplots(rows=5, cols=1, shared_xaxes=True, vertical_spacing=0.035,
                    subplot_titles=[f"{c}" + (f" — {share[c]:.0f}% of variance" if c in share else "") for c in comp.columns])
for i, (col, color) in enumerate(zip(comp.columns, colors), start=1):
    fig.add_trace(go.Scatter(x=sl.index, y=sl[col], mode="lines", line=dict(color=color, width=1.8), name=col,
                             fill="tozeroy" if i > 2 else None,
                             fillcolor=f"rgba{tuple(int(color[k:k + 2], 16) for k in (1, 3, 5)) + (0.12,)}",
                             hovertemplate="%{x|%a %d/%m %H:%M}: %{y:,.0f}<extra>" + col + "</extra>"), row=i, col=1)
fig.update_layout(title="Figure 24. MSTL decomposition over 3 weeks (5–25 Mar 2018): Observed = Trend + Daily + Weekly + Residual",
                  height=860, showlegend=False)
fig.show()

**Cách đọc Figure 24.** Cộng 4 hàng dưới theo từng thời điểm sẽ ra đúng hàng trên cùng. Hàng *Trend* gần như phẳng; hàng *Daily* lặp lại y hệt mỗi ngày; hàng *Weekly* nhô lên vào T2–T6 và lõm xuống vào T7–CN; hàng *Residual* nhỏ, chỉ có vài gai — đó là những giờ bất thường.

Vì mùa vụ lặp lại, có thể gom lại thành **một chu kỳ đại diện**: trung bình của thành phần mùa vụ ngày theo giờ, và của mùa vụ tuần theo (thứ, giờ):

In [59]:
s24 = comp["Daily seasonality (24h)"].groupby(comp.index.hour).mean()
s168 = comp["Weekly seasonality (168h)"].groupby(comp.index.dayofweek * 24 + comp.index.hour).mean()

fig = make_subplots(rows=1, cols=2, column_widths=[0.32, 0.68], horizontal_spacing=0.08,
                    subplot_titles=("One daily cycle (24h component)", "One weekly cycle (168h component)"))
fig.add_trace(go.Bar(x=s24.index, y=s24.values, marker_color=np.where(s24.values >= 0, AQUA, "#a9dcc8"),
                     hovertemplate="%{x}h: %{y:+,.0f}<extra></extra>"), row=1, col=1)
fig.add_trace(go.Scatter(x=s168.index, y=s168.values, mode="lines", line=dict(color=YELLOW, width=2),
                         fill="tozeroy", fillcolor="rgba(237,161,0,0.15)",
                         hovertemplate="hour %{x} of week: %{y:+,.0f}<extra></extra>"), row=1, col=2)
fig.update_xaxes(title_text="Hour of day", dtick=6, row=1, col=1)
fig.update_xaxes(tickvals=np.arange(12, 168, 24), ticktext=DAY_LABELS, row=1, col=2)
for d in range(1, 7):
    fig.add_vline(x=d * 24, line_color=GRID, line_width=1, row=1, col=2)
fig.update_yaxes(title_text="Deviation from trend (vehicles/hour)", col=1)
fig.update_layout(title="Figure 25. The two seasonal shapes MSTL extracted: the daily cycle swings ~5,000 vehicles, the weekly cycle cuts Sat/Sun mornings",
                  height=420, showlegend=False)
fig.show()

**Nhận xét.** Mùa vụ ngày giải thích khoảng **79%** phương sai, mùa vụ tuần thêm khoảng **15%**, xu hướng chỉ khoảng **1%** và phần dư khoảng **4%**. Độ mạnh mùa vụ ngày là 0,95, tuần là 0,80 (thang 0–1). Nói cách khác: **nếu biết "bây giờ là mấy giờ, thứ mấy" thì đã đoán được ~95% dao động của lưu lượng**; phần khó dự báo chỉ là 4% phần dư (sự kiện, thời tiết). Chuỗi gần như không có xu hướng dài hạn, nên **không cần sai phân (differencing)** để khử xu hướng cho các mô hình hồi quy; cái cần mô hình hóa là hai chu kỳ lồng nhau.

### 5.7 Tự tương quan (ACF / PACF)

**Ý tưởng.** *Tự tương quan* ở độ trễ (lag) $k$ đo xem **giá trị hiện tại giống giá trị cách đây $k$ giờ đến mức nào**:

$$\text{ACF}(k) = \text{corr}(y_t,\ y_{t-k})$$

Trước khi xem công thức tổng quát, hãy vẽ trực tiếp $y_t$ so với $y_{t-k}$ cho vài $k$. Nếu các điểm nằm sát đường chéo, giá trị $k$ giờ trước là một "dự báo" tốt cho hiện tại:

In [60]:
lag_list = [1, 3, 12, 24, 168]
lag_names = {1: "1h ago", 3: "3h ago", 12: "12h ago", 24: "yesterday", 168: "last week"}
edges = np.linspace(0, 7400, 61)
fig = make_subplots(rows=1, cols=len(lag_list), shared_yaxes=True, horizontal_spacing=0.025,
                    subplot_titles=[f"lag {k}: {lag_names[k]}<br>r = {y.autocorr(k):.2f}" for k in lag_list])
for c, k in enumerate(lag_list, start=1):
    pair = pd.DataFrame({"now": y, "past": y.shift(k)}).dropna()
    H, _, _ = np.histogram2d(pair["past"], pair["now"], bins=[edges, edges])
    centers = (edges[:-1] + edges[1:]) / 2
    fig.add_trace(go.Heatmap(z=np.log1p(H.T), x=centers, y=centers, colorscale=SEQ_BLUE, showscale=False,
                             customdata=H.T, hovertemplate="past %{x:,.0f}, now %{y:,.0f}: %{customdata:,.0f} hours<extra></extra>"),
                  row=1, col=c)
    fig.add_trace(go.Scatter(x=[0, 7400], y=[0, 7400], mode="lines", line=dict(color=RED, dash="dot", width=1),
                             hoverinfo="skip"), row=1, col=c)
    fig.update_xaxes(title_text="y(t−k)", range=[0, 7400], showticklabels=False, row=1, col=c)
fig.update_yaxes(title_text="y(t)", range=[0, 7400], col=1)
fig.update_layout(title="Figure 26. Lag plots (density): 12 hours ago is a poor guide, the same hour last week is the best",
                  height=380, showlegend=False, margin=dict(t=110))
fig.show()

**Đọc Figure 26.** Lag 1: điểm bám đường chéo (giờ liền trước gần giống giờ hiện tại). Lag 12: đám mây toả rộng, thậm chí lệch khỏi đường chéo — 12 giờ trước là "nửa kia" của ngày (đêm ↔ ngày). Lag 24 và nhất là lag 168 lại bám rất sát đường chéo — **cùng giờ hôm qua / tuần trước** là gợi ý mạnh nhất.

**ACF** chính là con số $r$ ở trên, tính cho mọi lag $k = 0, 1, 2, \dots$. **PACF** (tự tương quan *riêng phần*) trả lời câu hỏi tinh hơn: *sau khi đã biết các lag $1, \dots, k-1$, lag $k$ còn đem lại thông tin mới không?* Ví dụ ACF ở lag 2 cao chủ yếu vì lag 2 "đi qua" lag 1; PACF loại phần trung gian đó ra. Dải chấm xám là ngưỡng ý nghĩa 95% ($\pm 1{,}96/\sqrt{n}$): cột vượt ngưỡng là tương quan thật, không phải ngẫu nhiên.

Cách dùng trong thực tế:
- **ACF** → chọn các lag dài hạn cần đưa vào mô hình (đỉnh lặp lại = chu kỳ).
- **PACF** → chọn số lag ngắn hạn (bậc $p$ của thành phần AR trong ARIMA): PACF "tắt" sau lag $p$.

In [61]:
N_ACF, N_PACF = 24 * 15, 72
acf_vals = acf(y, nlags=N_ACF, fft=True)
pacf_vals = pacf(y, nlags=N_PACF, method="ywm")
band_ci = 1.96 / np.sqrt(len(y))
print(f"Ngưỡng ý nghĩa 95%: ±{band_ci:.3f}")

Ngưỡng ý nghĩa 95%: ±0.013


In [62]:
lags_acf = np.arange(N_ACF + 1)
acf_colors = np.where(lags_acf % 168 == 0, RED, np.where(lags_acf % 24 == 0, ORANGE, BLUE))
fig = make_subplots(rows=2, cols=1, vertical_spacing=0.14,
                    subplot_titles=("ACF up to 360 hours (15 days) — orange = multiples of 24h, red = multiples of 168h",
                                    "PACF up to 72 hours"))
fig.add_trace(go.Bar(x=lags_acf, y=acf_vals, marker_color=acf_colors, marker_line_width=0, name="ACF",
                     hovertemplate="lag %{x}: %{y:.3f}<extra></extra>"), row=1, col=1)
fig.add_trace(go.Bar(x=np.arange(N_PACF + 1), y=pacf_vals,
                     marker_color=np.where(np.abs(pacf_vals) > band_ci, ORANGE, MUTED), name="PACF",
                     hovertemplate="lag %{x}: %{y:.3f}<extra></extra>"), row=2, col=1)
for r in (1, 2):
    for s in (band_ci, -band_ci):
        fig.add_hline(y=s, line_dash="dot", line_color=MUTED, row=r, col=1)
for lag in (24, 168, 336):
    fig.add_annotation(x=lag, y=acf_vals[lag], text=f"lag {lag}: {acf_vals[lag]:.2f}", showarrow=True,
                       arrowcolor=MUTED, ay=-30, font_color=INK_2, row=1, col=1)
fig.add_annotation(x=12, y=acf_vals[12], text=f"lag 12: {acf_vals[12]:.2f}", showarrow=True, arrowcolor=MUTED,
                   ay=30, font_color=INK_2, row=1, col=1)
fig.update_xaxes(title_text="Lag (hours)", row=2, col=1)
fig.update_xaxes(tickvals=np.arange(0, N_ACF + 1, 24), row=1, col=1)
fig.update_layout(title="Figure 27. Autocorrelation: peaks every 24h and highest at 168h confirm the daily and weekly cycles",
                  height=680, showlegend=False, bargap=0.25)
fig.show()

In [63]:
lag_tbl = pd.DataFrame({"lag (giờ)": [1, 2, 3, 12, 24, 168],
                        "ACF": acf_vals[[1, 2, 3, 12, 24, 168]],
                        "PACF": [pacf_vals[k] if k <= N_PACF else np.nan for k in [1, 2, 3, 12, 24, 168]]})
lag_tbl

,lag (giờ),ACF,PACF
0,1,0.91,0.91
1,2,0.72,-0.67
2,3,0.51,0.18
3,12,-0.62,0.15
4,24,0.87,-0.10
5,168,0.94,NaN


**Nhận xét.** ACF có dạng **sóng lặp lại mỗi 24 giờ**, đáy âm ở lag 12 (ngày ↔ đêm), đỉnh ở **lag 24** và cao nhất ở **lag 168** — các đỉnh tuần (đỏ) cao hơn các đỉnh ngày xung quanh, nghĩa là "cùng giờ tuần trước" giống hiện tại hơn "cùng giờ hôm qua" (vì hôm qua có thể là cuối tuần). ACF **giảm rất chậm** qua 15 ngày — dấu hiệu của mùa vụ mạnh. PACF lớn ở lag 1 (0,91), âm mạnh ở lag 2 (−0,67), sau đó còn một cụm gai dương quanh lag 18–23 và âm ở lag 25 — dấu vết của chu kỳ ngày mà một mô hình AR ngắn không bắt được.

**Gợi ý cho bước tạo đặc trưng:** các lag `1, 2, 3, 24, 168` cùng trung bình trượt 24 giờ là bộ đặc trưng hợp lý cho mô hình hồi quy/cây. Với họ **ARIMA**, PACF gợi ý thành phần AR bậc khoảng 2–3 cho phần ngắn hạn, còn hai chu kỳ 24h/168h nên đưa vào dưới dạng mùa vụ hoặc biến Fourier (xem 5.8).

### 5.8 Tính dừng (stationarity)

Các mô hình họ ARIMA yêu cầu chuỗi (sau khi sai phân $d$ lần) là **dừng**: trung bình và phương sai không đổi theo thời gian. Hai kiểm định bổ sung cho nhau:

| Kiểm định | Giả thuyết $H_0$ | Kết luận "dừng" khi |
|---|---|---|
| **ADF** (Augmented Dickey–Fuller) | Chuỗi có nghiệm đơn vị (**không dừng**) | p < 0,05 → bác bỏ $H_0$ |
| **KPSS** | Chuỗi **dừng** quanh mức trung bình | p > 0,05 → không bác bỏ $H_0$ |

Kiểm định trên chuỗi gốc và trên chuỗi đã khử mùa vụ (Trend + Residual của MSTL).

In [64]:
def stationarity(series, name):
    adf_stat, adf_p, *_ = adfuller(series, maxlag=48, autolag="AIC")
    kpss_stat, kpss_p, *_ = kpss(series, regression="c", nlags="auto")
    return {"Chuỗi": name, "ADF stat": adf_stat, "ADF p-value": adf_p, "KPSS stat": kpss_stat,
            "KPSS p-value": kpss_p,
            "Kết luận": "Dừng" if (adf_p < 0.05 and kpss_p > 0.05) else ("Dừng quanh mùa vụ (ADF)" if adf_p < 0.05 else "Không dừng")}


deseason = comp["Trend"] + comp["Residual"]
pd.DataFrame([stationarity(y, "traffic_volume (gốc)"),
              stationarity(deseason, "Đã khử mùa vụ (Trend + Residual)"),
              stationarity(y.diff(168).dropna(), "Sai phân mùa vụ y(t) − y(t−168)")]).set_index("Chuỗi")

,ADF stat,ADF p-value,KPSS stat,KPSS p-value,Kết luận
Chuỗi,,,,,
traffic_volume (gốc),-21.81,0.00,0.56,0.03,Dừng quanh mùa vụ (ADF)
Đã khử mùa vụ (Trend + Residual),-14.22,0.00,0.75,0.01,Dừng quanh mùa vụ (ADF)
Sai phân mùa vụ y(t) − y(t−168),-16.06,0.00,0.01,0.10,Dừng


In [65]:
roll_w = 24 * 28
roll_mean, roll_std = y.rolling(roll_w, center=True).mean(), y.rolling(roll_w, center=True).std()
ds_mean, ds_std = deseason.rolling(roll_w, center=True).mean(), deseason.rolling(roll_w, center=True).std()
step = 24  # vẽ 1 điểm/ngày cho nhẹ

fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.05,
                    subplot_titles=("Original series", "Deseasonalized (Trend + Residual)"))
for c, (m, s) in enumerate([(roll_mean, roll_std), (ds_mean, ds_std)], start=1):
    m, s = m.iloc[::step].dropna(), s.iloc[::step].dropna()
    band(fig, m.index.to_numpy(), (m - s).values, (m + s).values, "rgba(42,120,214,0.15)", "±1 std", row=1, col=c)
    fig.add_trace(go.Scatter(x=m.index, y=m.values, mode="lines", line=dict(color=BLUE, width=2.5), name="Rolling mean",
                             hovertemplate="%{x|%d/%m/%Y}: %{y:,.0f}<extra>4-week mean</extra>"), row=1, col=c)
fig.update_yaxes(title_text="Vehicles/hour", col=1)
fig.update_layout(title="Figure 28. 4-week rolling mean ± std: level and spread stay roughly constant → no differencing for trend needed",
                  height=400, showlegend=False)
fig.show()

**Nhận xét.** Trung bình và độ lệch chuẩn trượt 4 tuần gần như **nằm ngang** suốt 2016–2018 ở cả chuỗi gốc lẫn chuỗi đã khử mùa vụ; Phần đã khử mùa vụ dao động hẹp hơn hẳn (dải ±1 std nhỏ). Về kiểm định:
- **ADF** bác bỏ nghiệm đơn vị cho cả ba chuỗi (p ≈ 0) → không có xu hướng ngẫu nhiên (random walk).
- **KPSS** vẫn bác bỏ tính dừng ở mức 5% cho chuỗi gốc (p = 0,03) và chuỗi đã khử mùa vụ (p = 0,01): mức nền trôi chậm theo mùa (thấp vào tháng 12–1, xem đường xanh) và KPSS rất nhạy khi $n$ lớn (~24 nghìn giờ).
- **Sai phân mùa vụ** $y_t - y_{t-168}$ đạt cả hai tiêu chí (ADF p ≈ 0, KPSS p = 0,10).

**Hệ quả cho ARIMA:** không cần sai phân thường ($d = 0$). Có hai cách xử lý mùa vụ: (1) sai phân mùa vụ theo tuần $y_t - y_{t-168}$ rồi fit ARIMA bậc thấp trên chuỗi đó; hoặc (2) mô hình hóa mùa vụ **tường minh** — bằng biến Fourier sin/cos chu kỳ 24h và 168h (SARIMAX với biến ngoại sinh, còn gọi là *dynamic harmonic regression*) hoặc ARIMA trên phần đã khử mùa vụ bởi MSTL — thay vì SARIMA với chu kỳ $m = 168$ (quá nặng để ước lượng trên ~24 nghìn giờ).

### 5.9 Phát hiện ngoại lai của `traffic_volume`: rolling mean/std so với rolling median/MAD

Áp rolling statistics trực tiếp lên chuỗi gốc không hiệu quả vì chu kỳ ngày làm độ lệch chuẩn bị "thổi phồng". Vì vậy:
1. Trừ đi **hồ sơ mùa vụ thứ × giờ** (trung vị) → phần dư $r_t$.
2. Chuẩn hóa phần dư trong cửa sổ trượt 1 tuần (168 giờ, đặt giữa), theo hai cách:
   - **Rolling mean/std:** $z_t = \dfrac{r_t - \text{mean}_{168}(r)}{\text{std}_{168}(r)}$
   - **Rolling median/MAD (robust):** $z_t = \dfrac{r_t - \text{median}_{168}(r)}{1{,}4826 \cdot \text{MAD}_{168}(r)}$ — hằng số 1,4826 đưa MAD về cùng thang với std khi dữ liệu chuẩn.
3. Đánh dấu ngoại lai khi $|z_t| > 3{,}5$ (ngưỡng Iglewicz–Hoaglin cho z-score robust).

**Câu hỏi:** cách nào phát hiện được các đoạn lưu lượng gần 0 kéo dài (đóng đường)?

**Bước 1 — phần dư sau khi trừ hồ sơ thứ × giờ:**

In [66]:
Z_THRESHOLD = 3.5
how = ts.index.dayofweek * 24 + ts.index.hour
profile = y.groupby(how).transform("median")
resid = y - profile
resid.describe().to_frame("resid")

,resid
count,"24,096.00"
mean,-46.35
std,494.53
min,"-5,544.00"
25%,-157.00
50%,0.00
75%,156.00
max,"2,447.00"


**Bước 2 — hai cách chuẩn hóa trong cửa sổ trượt 168 giờ:**

In [67]:
win = resid.rolling(168, center=True, min_periods=84)
z_std = (resid - win.mean()) / win.std()

In [68]:
mad = win.apply(lambda a: np.median(np.abs(a - np.median(a))), raw=True)
z_robust = (resid - win.median()) / (1.4826 * mad)

**Bước 3 — gắn cờ và so sánh.** "Giờ đóng đường" được định nghĩa: ban ngày 7h–19h, lưu lượng < 300 xe trong khi kỳ vọng > 3.000 xe.

In [69]:
flag_std, flag_robust = z_std.abs() > Z_THRESHOLD, z_robust.abs() > Z_THRESHOLD
ts["is_anomaly"] = flag_robust.astype(int)
closure = (y < 300) & (profile > 3000) & ts.index.hour.isin(range(7, 20))

compare = pd.DataFrame({
    "Phương pháp": ["Rolling mean/std", "Rolling median/MAD"],
    "Số giờ ngoại lai": [int(flag_std.sum()), int(flag_robust.sum())],
    "% số giờ": [f"{flag_std.mean():.2%}", f"{flag_robust.mean():.2%}"],
    f"Bắt được giờ đóng đường (/{int(closure.sum())})": [int((flag_std & closure).sum()), int((flag_robust & closure).sum())],
    "Trong đó là ngày lễ": [int(ts.loc[flag_std, "is_holiday"].sum()), int(ts.loc[flag_robust, "is_holiday"].sum())],
})
compare

,Phương pháp,Số giờ ngoại lai,% số giờ,Bắt được giờ đóng đường (/14),Trong đó là ngày lễ
0,Rolling mean/std,255,1.06%,0,52
1,Rolling median/MAD,1080,4.48%,14,202


In [70]:
seg = ts.loc["2016-07-04":"2016-07-31"]
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06, row_heights=[0.65, 0.35],
                    subplot_titles=("Actual vs. expected traffic", "z-scores of the residual: red = median/MAD, yellow dotted = mean/std (threshold ±3.5)"))
fig.add_trace(go.Scatter(x=seg.index, y=profile.loc[seg.index], mode="lines", name="Expected (weekday × hour median)",
                         line=dict(color=MUTED, width=1.5, dash="dot"), hovertemplate="%{y:,.0f}<extra>Expected</extra>"), row=1, col=1)
fig.add_trace(go.Scatter(x=seg.index, y=seg["traffic_volume"], mode="lines", name="Actual",
                         line=dict(color=BLUE, width=1.8), hovertemplate="%{y:,.0f}<extra>Actual</extra>"), row=1, col=1)
for name, flag, color, symbol in [("Rolling median/MAD", flag_robust, RED, "circle"),
                                  ("Rolling mean/std", flag_std, ORANGE, "x")]:
    pts = seg[flag.loc[seg.index]]
    fig.add_trace(go.Scatter(x=pts.index, y=pts["traffic_volume"], mode="markers",
                             name=f"{name}: {len(pts)} hours this month",
                             marker=dict(color=color, size=10, symbol=symbol, line=dict(color="#ffffff", width=1.5)),
                             hovertemplate="%{x|%a %d/%m %H:%M}: %{y:,.0f}<extra>" + name + "</extra>"), row=1, col=1)
fig.add_trace(go.Scatter(x=seg.index, y=z_robust.loc[seg.index].clip(-15, 15), mode="lines", name="z (median/MAD)",
                         line=dict(color=RED, width=1.6), showlegend=False,
                         hovertemplate="%{y:.1f}<extra>z median/MAD</extra>"), row=2, col=1)
fig.add_trace(go.Scatter(x=seg.index, y=z_std.loc[seg.index].clip(-15, 15), mode="lines", name="z (mean/std)",
                         line=dict(color=YELLOW, width=1.6, dash="dot"), showlegend=False,
                         hovertemplate="%{y:.1f}<extra>z mean/std</extra>"), row=2, col=1)
for s in (Z_THRESHOLD, -Z_THRESHOLD):
    fig.add_hline(y=s, line_dash="dot", line_color=INK_2, row=2, col=1)
n_std, n_rob = int((flag_std & closure).sum()), int((flag_robust & closure).sum())
fig.update_yaxes(title_text="Vehicles/hour", row=1, col=1)
fig.update_yaxes(title_text="z (clipped ±15)", row=2, col=1)
fig.update_layout(title=f"Figure 29. July 2016: rolling median/MAD flags {n_rob}/{int(closure.sum())} daytime road-closure hours, "
                        f"rolling mean/std flags {n_std}",
                  height=660, hovermode="x unified", margin=dict(t=130), legend=dict(y=1.08))
fig.show()

Ngoại lai phân bố thế nào trong cả giai đoạn? Đếm số giờ bị gắn cờ (median/MAD) theo từng ngày:

In [71]:
anom_day = ts["is_anomaly"].resample("D").sum()
kind = np.select([is_hol_day.reindex(anom_day.index).values,
                  (ts["weather_main"].eq("Snow").resample("D").sum() >= 6).reindex(anom_day.index).values],
                 ["Holiday", "Snow day (≥ 6 snowy hours)"], "Other day")
fig = go.Figure()
for name, color in [("Holiday", AQUA), ("Snow day (≥ 6 snowy hours)", BLUE), ("Other day", MUTED)]:
    m = kind == name
    fig.add_trace(go.Bar(x=anom_day.index[m], y=anom_day.values[m], name=name, marker_color=color,
                         marker_line_width=0, hovertemplate="%{x|%a %d/%m/%Y}: %{y} flagged hours<extra>" + name + "</extra>"))
fig.update_layout(title="Figure 30. Flagged hours per day: the tallest bars are holidays and snow days, i.e. real events",
                  yaxis_title="Anomalous hours in the day", height=420, barmode="overlay", bargap=0)
fig.show()

In [72]:
top = ts.loc[flag_robust].index.normalize().value_counts().head(10)
weather_day = ts.groupby(ts.index.normalize()).agg(holiday=("holiday", "first"),
                                                   snow_hours=("weather_main", lambda s: int((s == "Snow").sum())))
top_days = weather_day.loc[top.index].assign(**{"Số giờ ngoại lai": top.values})
top_days.index = top_days.index.strftime("%a %d/%m/%Y")
top_days.columns = ["Ngày lễ", "Số giờ có tuyết trong ngày", "Số giờ ngoại lai"]
top_days

,Ngày lễ,Số giờ có tuyết trong ngày,Số giờ ngoại lai
date_time,,,
Thu 11/01/2018,None,12,17
Mon 22/01/2018,None,19,15
Mon 19/02/2018,Washingtons Birthday,11,15
Mon 29/05/2017,Memorial Day,0,15
Sat 14/04/2018,None,20,14
Thu 23/11/2017,Thanksgiving Day,0,14
Fri 01/01/2016,New Years Day,3,14
Wed 04/07/2018,Independence Day,0,14
Sat 23/07/2016,None,0,14


**Kết luận.** Khi đường bị đóng nhiều giờ liền, chính các giờ đó làm **độ lệch chuẩn của cửa sổ phình to**, nên z-score theo mean/std nhỏ lại và ngoại lai "tự che" mình (masking effect) — thấy rõ ở hàng dưới Figure 29: ở đoạn đóng đường 23–24/7, đường đỏ lao xuống khoảng −8 trong khi đường vàng chỉ chạm khoảng −3 và không vượt ngưỡng. Median và MAD gần như không bị vài chục điểm cực đoan kéo lệch nên bắt được các đoạn này. Các ngày có nhiều giờ ngoại lai nhất là **ngày lễ** và **ngày bão tuyết** (Figure 30) — đều là sự kiện thật mà mô hình cần học.

**Xử lý.**
- **Không xóa / không thay thế** giá trị mục tiêu: đây là sự kiện có thật (đóng đường, ngày lễ, bão tuyết), không phải lỗi đo như `temp = 0 K`. Sửa chúng trên tập test sẽ làm điểm đánh giá đẹp giả tạo.
- **Giữ cờ `is_anomaly`** (theo median/MAD) để phân tích sai số ở phần mô hình hóa (mô hình sai nhiều ở giờ nào?). Cờ này **không** dùng làm đặc trưng đầu vào vì được tính bằng cửa sổ đặt giữa (nhìn cả tương lai) → sẽ rò rỉ dữ liệu.

<a id="6"></a>
## 6. Chuẩn bị dữ liệu cho mô hình

### 6.1 Tạo đặc trưng

| Nhóm | Đặc trưng | Ghi chú |
|---|---|---|
| Lag | `lag_1`, `lag_2`, `lag_3`, `lag_24`, `lag_168` | Từ ACF/PACF ở mục 5.7 |
| Rolling | `roll_mean_24`, `roll_std_24` | Tính trên chuỗi đã `shift(1)` → chỉ dùng quá khứ |
| Lịch | `hour`, `dayofweek`, `month`, `is_weekend`, `is_holiday` | `hour`, `dayofweek` mã hóa thêm dạng sin/cos để 23h nằm cạnh 0h |
| Thời tiết | `temp_c`, `rain_1h`, `snow_1h`, `clouds_all`, `n_weather`, `weather_group` | `weather_group` = `weather_main`, gộp loại hiếm vào `Other` |

Thời tiết được dùng ở **cùng giờ dự báo** — giả định có dự báo thời tiết cho giờ tới (đây là giả định phổ biến trong bài toán dự báo nhu cầu, cần nêu trong báo cáo).

In [73]:
LAGS = [1, 2, 3, 24, 168]
WEATHER_NUM = ["temp_c", "rain_1h", "snow_1h", "clouds_all", "n_weather"]


def add_lag_features(f, yv):
    for k in LAGS:
        f[f"lag_{k}"] = yv.shift(k)
    past = yv.shift(1)
    f["roll_mean_24"] = past.rolling(24).mean()
    f["roll_std_24"] = past.rolling(24).std()


def add_calendar_features(f, ts):
    f["hour"], f["dayofweek"], f["month"] = ts.index.hour, ts.index.dayofweek, ts.index.month
    f["hour_sin"], f["hour_cos"] = np.sin(2 * np.pi * f["hour"] / 24), np.cos(2 * np.pi * f["hour"] / 24)
    f["dow_sin"], f["dow_cos"] = np.sin(2 * np.pi * f["dayofweek"] / 7), np.cos(2 * np.pi * f["dayofweek"] / 7)
    f["is_weekend"] = (f["dayofweek"] >= 5).astype(int)
    f["is_holiday"] = ts["is_holiday"]


def add_weather_features(f, ts):
    for c in WEATHER_NUM:
        f[c] = ts[c]
    counts = ts["weather_main"].value_counts()
    rare = counts[counts < 100].index
    f["weather_group"] = ts["weather_main"].where(~ts["weather_main"].isin(rare), "Other")


def make_features(ts):
    f = pd.DataFrame(index=ts.index)
    add_lag_features(f, ts["traffic_volume"])
    add_calendar_features(f, ts)
    add_weather_features(f, ts)
    f["traffic_volume"] = ts["traffic_volume"]
    return f.dropna()

In [74]:
feat = make_features(ts)
print(f"Số dòng sau khi bỏ {len(ts) - len(feat)} giờ đầu chưa đủ lag_168: {len(feat):,}")
print("weather_group:", feat["weather_group"].value_counts().to_dict())

Số dòng sau khi bỏ 168 giờ đầu chưa đủ lag_168: 23,928
weather_group: {'Clear': 8218, 'Clouds': 8025, 'Rain': 2762, 'Mist': 1942, 'Snow': 1586, 'Haze': 494, 'Drizzle': 387, 'Thunderstorm': 370, 'Fog': 133, 'Other': 11}


In [75]:
feat.head()

,lag_1,lag_2,lag_3,lag_24,lag_168,roll_mean_24,roll_std_24,hour,dayofweek,month,hour_sin,hour_cos,dow_sin,dow_cos,is_weekend,is_holiday,temp_c,rain_1h,snow_1h,clouds_all,n_weather,weather_group,traffic_volume
date_time,,,,,,,,,,,,,,,,,,,,,,,
2016-01-08 00:00:00,"1,241.00","2,075.50","2,910.00",558.00,"1,513.00","3,544.38","2,016.14",0,4,1,0.00,1.00,-0.43,-0.90,0,0,0.20,0.00,0.06,75.00,4,Snow,628.00
2016-01-08 01:00:00,628.00,"1,241.00","2,075.50",323.00,"1,550.00","3,547.29","2,011.67",1,4,1,0.26,0.97,-0.43,-0.90,0,0,0.13,0.00,0.17,90.00,3,Snow,359.00
2016-01-08 02:00:00,359.00,628.00,"1,241.00",315.00,"1,134.50","3,548.79","2,009.18",2,4,1,0.50,0.87,-0.43,-0.90,0,0,0.10,0.00,0.12,90.00,1,Snow,380.00
2016-01-08 03:00:00,380.00,359.00,628.00,524.00,719.00,"3,551.50","2,004.67",3,4,1,0.71,0.71,-0.43,-0.90,0,0,0.07,0.00,0.08,90.00,2,Snow,401.00
2016-01-08 04:00:00,401.00,380.00,359.00,"1,081.00",533.00,"3,546.38","2,012.89",4,4,1,0.87,0.50,-0.43,-0.90,0,0,-0.13,0.00,0.00,90.00,2,Snow,815.00


Vì sao mã hóa giờ bằng sin/cos? Với số nguyên `hour`, 23h và 0h cách nhau 23 đơn vị dù thực tế chỉ cách 1 giờ. Đặt giờ lên đường tròn đơn vị thì khoảng cách được bảo toàn:

In [76]:
hh = np.arange(24)
fig = go.Figure(go.Scatter(x=np.sin(2 * np.pi * hh / 24), y=np.cos(2 * np.pi * hh / 24), mode="markers+text",
                           text=[f"{h}h" for h in hh], textposition="middle right", textfont_color=INK_2,
                           marker=dict(size=14, color=hour_mean.values, colorscale=SEQ_BLUE, showscale=True,
                                       colorbar=dict(title="Mean<br>vehicles"), line=dict(color=INK_2, width=1)),
                           hovertemplate="%{text}: sin=%{x:.2f}, cos=%{y:.2f}<extra></extra>"))
fig.update_layout(title="Figure 31. Cyclical encoding: 23h sits next to 0h on the (hour_sin, hour_cos) circle",
                  xaxis=dict(title="hour_sin", range=[-1.4, 1.5], scaleanchor="y"), yaxis=dict(title="hour_cos", range=[-1.3, 1.3]),
                  height=520, width=620)
fig.show()

### 6.2 Chia train / validation / test theo thời gian

**Không xáo trộn**: mô hình chỉ được học từ quá khứ để dự báo tương lai. Nếu chia ngẫu nhiên, `lag_1` của một giờ trong test chính là giá trị mục tiêu của một giờ nằm trong train → rò rỉ dữ liệu và điểm số ảo.

| Tập | Giai đoạn |
|---|---|
| Train | 08/01/2016 – 30/11/2017 |
| Validation | 01/12/2017 – 30/04/2018 |
| Test | 01/05/2018 – 30/09/2018 |

Khi tinh chỉnh siêu tham số ở phần mô hình hóa sẽ dùng `TimeSeriesSplit` trên tập train.

In [77]:
TRAIN_END, VAL_END = "2017-11-30 23:00", "2018-04-30 23:00"
train = feat.loc[:TRAIN_END]
val = feat.loc[pd.Timestamp(TRAIN_END) + pd.Timedelta(hours=1):VAL_END]
test = feat.loc[pd.Timestamp(VAL_END) + pd.Timedelta(hours=1):]
assert train.index.max() < val.index.min() and val.index.max() < test.index.min()

In [78]:
split_tbl = pd.DataFrame({
    "Tập": ["Train", "Validation", "Test"],
    "Từ": [d.index.min().strftime("%d/%m/%Y %H:%M") for d in (train, val, test)],
    "Đến": [d.index.max().strftime("%d/%m/%Y %H:%M") for d in (train, val, test)],
    "Số giờ": [len(d) for d in (train, val, test)],
    "Tỷ lệ": [f"{len(d) / len(feat):.1%}" for d in (train, val, test)],
    "Trung bình xe/giờ": [d["traffic_volume"].mean() for d in (train, val, test)],
    "% giờ nội suy": [f"{ts.loc[d.index, 'is_imputed'].mean():.1%}" for d in (train, val, test)],
    "% giờ có tuyết": [f"{(d['weather_group'] == 'Snow').mean():.1%}" for d in (train, val, test)],
})
split_tbl

,Tập,Từ,Đến,Số giờ,Tỷ lệ,Trung bình xe/giờ,% giờ nội suy,% giờ có tuyết
0,Train,08/01/2016 00:00,30/11/2017 23:00,16632,69.5%,"3,322.07",5.6%,5.7%
1,Validation,01/12/2017 00:00,30/04/2018 23:00,3624,15.1%,"3,240.12",0.5%,17.6%
2,Test,01/05/2018 00:00,30/09/2018 23:00,3672,15.3%,"3,358.08",0.2%,0.0%


In [79]:
fig = go.Figure()
for name, part, color in [("Train", train, BLUE), ("Validation", val, ORANGE), ("Test", test, AQUA)]:
    dm = part["traffic_volume"].resample("D").mean()
    fig.add_trace(go.Scatter(x=dm.index, y=dm.values, mode="lines", name=f"{name} ({len(part):,} hours)",
                             line=dict(color=color, width=1.6), hovertemplate="%{x|%d/%m/%Y}: %{y:,.0f}<extra>" + name + "</extra>"))
for cut in (TRAIN_END, VAL_END):
    fig.add_vline(x=pd.Timestamp(cut), line_dash="dot", line_color=INK_2)
fig.update_layout(title="Figure 32. Chronological split: train → validation → test, no overlap",
                  yaxis_title="Vehicles/hour (daily mean)", height=420, hovermode="x unified")
fig.show()

In [80]:
fig = go.Figure()
for name, part, color in [("Train", train, BLUE), ("Validation", val, ORANGE), ("Test", test, AQUA)]:
    counts, e = np.histogram(part["traffic_volume"], bins=np.arange(0, 7600, 200), density=True)
    fig.add_trace(go.Scatter(x=(e[:-1] + e[1:]) / 2, y=counts * 200 * 100, mode="lines", name=name,
                             line=dict(color=color, width=2.5, shape="spline"),
                             hovertemplate="%{x:,.0f}: %{y:.1f}% of hours<extra>" + name + "</extra>"))
fig.update_layout(title="Figure 33. Target distribution per split: the three sets share the same bimodal shape",
                  xaxis_title="Vehicles/hour", yaxis_title="% of hours (200-vehicle bins)", height=400)
fig.show()

**Lưu ý.** Tập validation rơi vào mùa đông – xuân (có tuyết), còn tập test là mùa hè – thu (không có tuyết). Lưu lượng trung bình và hình dạng phân phối của ba tập rất gần nhau (Figure 33), nên chênh lệch phân phối không lớn; tuy vậy cần nhắc trong báo cáo rằng điểm trên test phản ánh mùa hè. Tập train chứa phần lớn giờ được nội suy (5,6%), val/test gần như toàn dữ liệu thật (≤ 0,5%) → đánh giá trên dữ liệu thật.

### 6.3 Encoding và scaling (fit trên **train**, áp dụng cho val/test)

- **Encoding:** `weather_group` → `OneHotEncoder(handle_unknown="ignore")` (loại thời tiết chưa thấy ở train sẽ thành vector 0, không làm lỗi). `hour`, `dayofweek` đã có bản sin/cos; `is_weekend`, `is_holiday` là nhị phân sẵn.
- **Scaling:** `StandardScaler` cho các cột số. Scaler **chỉ fit trên train** — nếu fit trên toàn bộ dữ liệu, trung bình/độ lệch chuẩn của tương lai sẽ lọt vào quá khứ. Cần cho Ridge/Linear Regression; các mô hình cây (Random Forest, Gradient Boosting) không bị ảnh hưởng bởi scaling.

In [81]:
TARGET = "traffic_volume"
CAT_COLS = ["weather_group"]
NUM_SCALE = [f"lag_{k}" for k in LAGS] + ["roll_mean_24", "roll_std_24"] + WEATHER_NUM + ["month"]
PASS_COLS = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "is_weekend", "is_holiday"]

In [82]:
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(train[CAT_COLS])
scaler = StandardScaler().fit(train[NUM_SCALE])


def transform(part):
    num = pd.DataFrame(scaler.transform(part[NUM_SCALE]), columns=NUM_SCALE, index=part.index)
    cat = pd.DataFrame(ohe.transform(part[CAT_COLS]), columns=ohe.get_feature_names_out(CAT_COLS), index=part.index)
    return pd.concat([num, part[PASS_COLS], cat], axis=1)

In [83]:
X_train, X_val, X_test = transform(train), transform(val), transform(test)
y_train, y_val, y_test = train[TARGET], val[TARGET], test[TARGET]
print("X_train:", X_train.shape, "| X_val:", X_val.shape, "| X_test:", X_test.shape)
print("Cột one-hot:", list(ohe.get_feature_names_out(CAT_COLS)))

X_train: (16632, 29) | X_val: (3624, 29) | X_test: (3672, 29)
Cột one-hot: ['weather_group_Clear', 'weather_group_Clouds', 'weather_group_Drizzle', 'weather_group_Fog', 'weather_group_Haze', 'weather_group_Mist', 'weather_group_Other', 'weather_group_Rain', 'weather_group_Snow', 'weather_group_Thunderstorm']


In [84]:
X_train.describe().T[["mean", "std", "min", "max"]].head(len(NUM_SCALE))

,mean,std,min,max
lag_1,0.00,1.00,-1.69,2.02
lag_2,-0.00,1.00,-1.69,2.02
lag_3,0.00,1.00,-1.69,2.02
lag_24,0.00,1.00,-1.69,2.02
lag_168,-0.00,1.00,-1.69,2.02
roll_mean_24,0.00,1.00,-6.44,1.74
roll_std_24,0.00,1.00,-5.09,2.10
temp_c,0.00,1.00,-3.36,2.11
rain_1h,-0.00,1.00,-0.08,56.76
snow_1h,0.00,1.00,-0.02,77.23


In [85]:
fig = make_subplots(rows=2, cols=2, vertical_spacing=0.18, horizontal_spacing=0.1,
                    subplot_titles=("temp_c — before scaling (°C)", "temp_c — after scaling",
                                    "lag_24 — before scaling (vehicles/hour)", "lag_24 — after scaling"))
for r, col in enumerate(["temp_c", "lag_24"], start=1):
    fig.add_trace(hist_bar(train[col].to_numpy(), bins=50, color=BLUE), row=r, col=1)
    fig.add_trace(hist_bar(X_train[col].to_numpy(), bins=50, color=ORANGE), row=r, col=2)
fig.update_yaxes(title_text="Hours", col=1)
fig.update_layout(title="Figure 34. StandardScaler only changes the scale (mean 0, std 1); the distribution shape is unchanged",
                  height=560, showlegend=False)
fig.show()

In [86]:
pd.DataFrame({"trước: mean": train[["temp_c", "lag_24"]].mean(), "trước: std": train[["temp_c", "lag_24"]].std(),
              "sau: mean": X_train[["temp_c", "lag_24"]].mean(), "sau: std": X_train[["temp_c", "lag_24"]].std()}).round(3)

,trước: mean,trước: std,sau: mean,sau: std
temp_c,9.53,11.68,0.00,1.00
lag_24,"3,321.64","1,962.43",0.00,1.00


### 6.4 Tương quan giữa các đặc trưng và mục tiêu (tập train)

In [87]:
corr_cols = [TARGET] + [f"lag_{k}" for k in LAGS] + ["roll_mean_24", "hour_sin", "hour_cos", "is_weekend",
                                                      "is_holiday", "temp_c", "rain_1h", "snow_1h", "clouds_all"]
corr = train[corr_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
z = corr.where(~mask)

fig = go.Figure(go.Heatmap(z=z.values, x=corr.columns, y=corr.columns, zmin=-1, zmax=1, colorscale=DIVERGING,
                           text=z.round(2).astype(object).where(z.notna(), "").values, texttemplate="%{text}",
                           textfont=dict(size=10),
                           xgap=2, ygap=2, colorbar=dict(title="r"),
                           hovertemplate="%{y} × %{x}: %{z:.2f}<extra></extra>"))
fig.update_layout(title="Figure 35. Pearson correlation on the training set (lower triangle)",
                  height=640, yaxis_autorange="reversed", xaxis_showgrid=False, yaxis_showgrid=False)
fig.show()

In [88]:
r_target = corr[TARGET].drop(TARGET).sort_values(key=np.abs)
group_color = {**{c: BLUE for c in r_target.index if c.startswith(("lag", "roll"))},
               **{c: ORANGE for c in ["hour_sin", "hour_cos", "is_weekend", "is_holiday"]},
               **{c: AQUA for c in ["temp_c", "rain_1h", "snow_1h", "clouds_all"]}}
fig = go.Figure(go.Bar(x=r_target.values, y=r_target.index, orientation="h",
                       marker_color=[group_color[c] for c in r_target.index],
                       text=[f"{v:+.2f}" for v in r_target.values], textposition="outside", textfont_color=INK_2,
                       showlegend=False, hovertemplate="%{y}: r = %{x:+.2f}<extra></extra>"))
for name, color in [("Lag / rolling", BLUE), ("Calendar", ORANGE), ("Weather", AQUA)]:
    fig.add_trace(go.Bar(x=[None], y=[None], marker_color=color, name=name, showlegend=True))
fig.add_vline(x=0, line_color=INK_2, line_width=1)
fig.update_layout(title="Figure 36. Correlation with traffic_volume: lags ≫ calendar ≫ weather",
                  xaxis=dict(title="Pearson r with target", range=[-1.1, 1.15]), height=480, barmode="overlay")
fig.show()

**Kết luận.** `lag_168` (r = 0,95), `lag_1` (0,91) và `lag_24` (0,87) tương quan mạnh nhất với mục tiêu, tiếp theo là `hour_cos` (−0,77). Các biến thời tiết đều có |r| ≤ 0,12. Lưu ý tương quan Pearson chỉ đo quan hệ tuyến tính: ảnh hưởng của tuyết (−10,6% ở mục 5.5) là phi tuyến và hiếm, nên r nhỏ không có nghĩa là vô dụng — mô hình cây có thể khai thác được. Ngoài ra các lag 1–3 tương quan rất mạnh với nhau (đa cộng tuyến) → nên dùng **Ridge** thay cho hồi quy tuyến tính thường.

<a id="7"></a>
## 7. Tổng kết EDA

**Dữ liệu và làm sạch**

| Vấn đề phát hiện | Số liệu | Cách xử lý |
|---|---|---|
| Dòng trùng mốc giờ (1 giờ có nhiều hiện tượng thời tiết) | 7.629 / 48.204 dòng (15,8%) | Gộp còn 1 dòng/giờ, thêm `n_weather` |
| Giờ bị thiếu (toàn bộ 2012–2018) | 11.976 giờ (22,8%), gap dài nhất 7.386 giờ ≈ 10 tháng | Chỉ dùng 01/2016 – 09/2018 (95,8% giờ có dữ liệu, gap dài nhất 9 giờ) |
| Giờ thiếu trong giai đoạn chọn | 1.012 / 24.096 giờ (4,2%) | Reindex theo giờ, nội suy tuyến tính theo thời gian, cờ `is_imputed` |
| Lỗi cảm biến | 10 giờ `temp = 0 K`, 1 giờ `rain_1h = 9.831 mm` | Đổi thành NaN rồi nội suy |
| Nhãn ngày lễ chỉ ở 00:00 | 61 dòng | Lan nhãn cho cả 24 giờ của ngày |
| Ngoại lai của `traffic_volume` | 1.080 giờ (4,5%) theo median/MAD | Giữ nguyên giá trị (sự kiện thật), cờ `is_anomaly` để phân tích lỗi |

**Những điều EDA trả lời được (có số đối chiếu)**
1. **Chu kỳ ngày và tuần chi phối chuỗi:** MSTL cho thấy mùa vụ ngày giải thích ≈ 79% phương sai, mùa vụ tuần ≈ 15%, xu hướng chỉ ≈ 1%. Giờ đông nhất (16h) gấp ≈ 14,5 lần giờ vắng nhất (3h), trong khi chênh lệch giữa các tháng chỉ ≈ 12%; hình dạng ngày gần như không đổi theo mùa.
2. **Ngày lễ giảm ≈ 28% tổng lưu lượng ngày**, tương đương cuối tuần (−26%), nhưng rất khác nhau giữa các lễ: lễ lớn giảm 33–49%, Columbus/Veterans Day gần như không giảm, State Fair còn tăng nhẹ → cần `is_holiday`, tốt hơn nữa là phân biệt lễ lớn/nhỏ.
3. **Thời tiết chỉ quan trọng khi cực đoan:** so với cùng thứ-cùng giờ, tuyết làm giảm ≈ 10,6% (tuyết lớn ≈ 43%) và mưa/tuyết "heavy" ≈ 13,6%; các loại thời tiết khác lệch dưới 3%.
4. **Lag mạnh nhất là cùng giờ tuần trước:** `lag_168` có r = 0,95, `lag_1` 0,91, `lag_24` 0,87 → baseline naive "bằng giá trị tuần trước" sẽ là mốc so sánh khó vượt.
5. **Không có xu hướng ngẫu nhiên, mùa vụ mạnh:** ADF bác bỏ nghiệm đơn vị, trung bình/độ lệch chuẩn trượt gần như nằm ngang → $d = 0$; sai phân mùa vụ 168 giờ cho chuỗi dừng theo cả ADF lẫn KPSS.
6. **Phát hiện ngoại lai cần thống kê robust:** rolling median/MAD bắt được 14/14 giờ đóng đường ban ngày, rolling mean/std bắt được 0/14 do hiệu ứng che lấp (masking).

**Đầu vào cho phần mô hình hóa:** `X_train` (16.632 × 29), `X_val` (3.624 × 29), `X_test` (3.672 × 29), chia theo thời gian, encoder/scaler chỉ fit trên train.

**Bước tiếp theo**
1. Baseline naive: `lag_1`, `lag_24`, `lag_168` và trung bình mùa vụ thứ × giờ.
2. Mô hình thống kê: **SARIMAX với biến Fourier** (sin/cos chu kỳ 24h và 168h) + biến ngoại sinh `is_holiday`, tuyết; phần ARIMA bậc thấp ($p \approx 2$–$3$, $d = 0$) cho phần dư ngắn hạn — hoặc ARIMA trên chuỗi đã sai phân mùa vụ $y_t - y_{t-168}$ / đã khử mùa vụ bằng MSTL. Không dùng SARIMA $m = 168$ trực tiếp (quá nặng).
3. Mô hình học máy: Ridge, Random Forest, Gradient Boosting trên bộ đặc trưng ở mục 6.
4. So sánh MAE, RMSE, MAPE trên validation **cùng tầm dự báo** (vd. 1 giờ tới, cập nhật dần), chọn mô hình rồi báo cáo trên test; vẽ dự báo vs thực tế và phân tích sai số ở các giờ `is_anomaly`.

**Hạn chế:** chỉ có một trạm đo và một chiều đi (hướng tây); thời tiết được dùng như "biết trước" cho giờ dự báo; tập test chỉ gồm mùa hè – thu 2018.